In [ ]:

# =============================================================================
# LSTM V2 COMPETING-RISK — CELL 1
# ENVIRONMENT + FROZEN DATASET INTEGRITY
#
# NO TRAINING. NO OFFICIAL VALIDATION OR TEST ACCESS.
# =============================================================================

from pathlib import Path
import hashlib
import json
import platform
import random

import numpy as np
import pandas as pd
import torch

from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/network-attack-forecasting")
DATA_DIR = ROOT / "Synthetic_NAF_v2/release_candidate_corrected"
FREEZE_DIR = ROOT / "Synthetic_NAF_v2/FINAL_RELEASE_FREEZE"
MODEL_ROOT = ROOT / "Synthetic_NAF_v2/models/LSTM_V2_CompetingRisk"

TRAIN_FILE = DATA_DIR / "windows_W10_TRAIN.npz"
META_FILE = DATA_DIR / "episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv"
CONTRACT_FILE = DATA_DIR / "FEATURE_AND_DATASET_CONTRACT.json"
MANIFEST_FILE = FREEZE_DIR / "Synthetic_NAF_v2_RELEASE_MANIFEST.json"
LOCK_FILE = FREEZE_DIR / "Synthetic_NAF_v2_FREEZE_LOCK.json"
HASH_FILE = DATA_DIR / "SHA256SUMS.json"

SEED = 20261024
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_sha256(obj):
    content = json.dumps(
        obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False
    ).encode("utf-8")
    return hashlib.sha256(content).hexdigest()


for path in [
    TRAIN_FILE, META_FILE, CONTRACT_FILE,
    MANIFEST_FILE, LOCK_FILE, HASH_FILE
]:
    if not path.is_file():
        raise FileNotFoundError(path)

CONTRACT = load_json(CONTRACT_FILE)
MANIFEST = load_json(MANIFEST_FILE)
FREEZE_LOCK = load_json(LOCK_FILE)

assert MANIFEST["release_version"] == "2.0.0"
assert (
    MANIFEST["release_status"]
    == "FROZEN_CONTROLLED_SYNTHETIC_BENCHMARK"
)
assert (
    canonical_sha256(MANIFEST)
    == FREEZE_LOCK["manifest_sha256_canonical"]
)

# Verify hashes from both existing release records.
for filename, expected in load_json(HASH_FILE).items():
    if sha256_file(DATA_DIR / filename) != expected:
        raise RuntimeError(f"Release hash mismatch: {filename}")

for filename, expected in MANIFEST["data_integrity"]["file_sha256"].items():
    if sha256_file(DATA_DIR / filename) != expected:
        raise RuntimeError(f"Manifest hash mismatch: {filename}")

FEATURES = list(CONTRACT["features"])
FAMILIES = list(CONTRACT["attack_families"])

assert len(FEATURES) == 14
assert len(FAMILIES) == 5
assert CONTRACT["history_minutes"] == 10
assert CONTRACT["forecast_horizons"] == [5, 10, 15]

FAMILY_TO_ID = {
    name: index + 1 for index, name in enumerate(FAMILIES)
}
ID_TO_FAMILY = {
    index + 1: name for index, name in enumerate(FAMILIES)
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 90)
print("LSTM V2 COMPETING-RISK — RELEASE VERIFIED")
print("=" * 90)
print("Release:", MANIFEST["release_version"])
print("PyTorch:", torch.__version__)
print("Python:", platform.python_version())
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("Features:", len(FEATURES))
print("Attack families:", FAMILIES)
print("Dataset integrity: PASS")
print("Training: NOT STARTED")
print("Official VALIDATION/TEST: NOT LOADED")


In [ ]:

# =============================================================================
# LSTM V2 COMPETING-RISK — CELL 2
# LOAD TRAIN ARRAYS AND CONSTRUCT 76-CLASS TARGET
#
# NO TRAINING
# =============================================================================

with np.load(TRAIN_FILE, allow_pickle=False) as pack:
    X_RAW = pack["X"].astype(np.float32)
    Y_HORIZONS = pack["y_horizons"].astype(np.int8)
    ONSET = pack["onset_class"].astype(np.int16)
    FAMILY = pack["family_class"].astype(np.int16)
    EPISODE_ID = pd.Series(
        pack["episode_id"]
    ).astype(str).to_numpy()
    MINUTE = pack["minute"].astype(np.int16)

assert X_RAW.shape == (88186, 10, 14)
assert Y_HORIZONS.shape == (88186, 3)
assert len(ONSET) == len(X_RAW)
assert len(FAMILY) == len(X_RAW)
assert np.isfinite(X_RAW).all()
assert (X_RAW >= 0).all()

assert np.all((ONSET >= 0) & (ONSET <= 15))
assert np.all((FAMILY >= 0) & (FAMILY <= 5))
assert np.array_equal(ONSET == 0, FAMILY == 0)

for col, horizon in enumerate([5, 10, 15]):
    expected = ((ONSET >= 1) & (ONSET <= horizon)).astype(np.int8)
    assert np.array_equal(Y_HORIZONS[:, col], expected)

# Important: class 0 means no event within 15 minutes.
JOINT_TARGET = np.where(
    ONSET == 0,
    0,
    (ONSET - 1) * 5 + FAMILY
).astype(np.int64)

assert JOINT_TARGET.min() == 0
assert JOINT_TARGET.max() <= 75

# Decode and recover original onset/family labels.
decoded_onset = np.where(
    JOINT_TARGET == 0,
    0,
    (JOINT_TARGET - 1) // 5 + 1
)

decoded_family = np.where(
    JOINT_TARGET == 0,
    0,
    (JOINT_TARGET - 1) % 5 + 1
)

assert np.array_equal(decoded_onset, ONSET)
assert np.array_equal(decoded_family, FAMILY)

# Verify no hidden missing joint outcomes.
joint_counts = np.bincount(JOINT_TARGET, minlength=76)
assert len(joint_counts) == 76
assert np.all(joint_counts > 0), (
    "One or more joint classes have no TRAIN examples."
)

# Decode the joint label back to cumulative horizon labels.
for col, horizon in enumerate([5, 10, 15]):
    recovered = (
        (decoded_onset >= 1) & (decoded_onset <= horizon)
    ).astype(np.int8)
    assert np.array_equal(recovered, Y_HORIZONS[:, col])

META = pd.read_csv(META_FILE)
META["Episode_ID"] = META["Episode_ID"].astype(str)
assert META["Episode_ID"].is_unique

TRAIN_META = META.loc[
    META["Split"].astype(str) == "TRAIN"
].set_index("Episode_ID")

assert len(np.unique(EPISODE_ID)) == 1400
assert set(np.unique(EPISODE_ID)) == set(TRAIN_META.index)

SOURCE = TRAIN_META.loc[
    EPISODE_ID, "Source"
].astype(str).to_numpy()

CAL_GROUP = TRAIN_META.loc[
    EPISODE_ID, "Calibration_Group_ID"
].astype(str).to_numpy()

CV_GROUPS = np.array([
    f"{source}::{group}"
    for source, group in zip(SOURCE, CAL_GROUP)
])

assert len(np.unique(CV_GROUPS)) == 35
assert set(np.unique(SOURCE)) == {"CIC_IDS2017", "UNSW_NB15"}

print("=" * 90)
print("COMPETING-RISK TARGETS VERIFIED")
print("=" * 90)
print("TRAIN sequences:", X_RAW.shape)
print("Joint classes:", len(joint_counts))
print("Observed joint classes:", int((joint_counts > 0).sum()))
print("Class 0 windows:", int(joint_counts[0]))
print("Class 0 prevalence:", round(joint_counts[0] / len(X_RAW), 6))
print("Event within 15 min:", int((JOINT_TARGET > 0).sum()))
print("Unique episodes:", len(np.unique(EPISODE_ID)))
print("Calibration groups:", len(np.unique(CV_GROUPS)))

print("\nFirst 10 encoded examples:")
print(pd.DataFrame({
    "onset_minute": ONSET[:10],
    "family_id": FAMILY[:10],
    "joint_class": JOINT_TARGET[:10]
}).to_string(index=False))

print("\nPer-family event-window counts:")
for fid, name in ID_TO_FAMILY.items():
    count = int(np.sum(FAMILY == fid))
    print(f"  {fid}: {name}: {count:,}")

print("\n76-class decoding: PASS")
print("H5/H10/H15 target reconstruction: PASS")
print("Training: NOT STARTED")


In [ ]:

# =============================================================================
# LSTM V2 COMPETING-RISK — CELL 3
# VERIFY TRAIN-ONLY FIVE-FOLD GROUP CV
#
# NO TRAINING
# =============================================================================

from sklearn.model_selection import GroupKFold

CV_FOLDS = list(
    GroupKFold(n_splits=5).split(
        X_RAW,
        JOINT_TARGET,
        groups=CV_GROUPS
    )
)

FOLD_ASSIGNMENTS = np.full(len(X_RAW), -1, dtype=np.int8)
fold_rows = []

for fold, (fit_idx, hold_idx) in enumerate(CV_FOLDS, start=1):
    fit_groups = set(CV_GROUPS[fit_idx])
    hold_groups = set(CV_GROUPS[hold_idx])

    fit_episodes = set(EPISODE_ID[fit_idx])
    hold_episodes = set(EPISODE_ID[hold_idx])

    assert fit_groups.isdisjoint(hold_groups)
    assert fit_episodes.isdisjoint(hold_episodes)

    assert len(hold_groups) == 7
    assert len(fit_groups) == 28

    assert set(SOURCE[fit_idx]) == {"CIC_IDS2017", "UNSW_NB15"}
    assert set(SOURCE[hold_idx]) == {"CIC_IDS2017", "UNSW_NB15"}

    fit_classes = np.bincount(JOINT_TARGET[fit_idx], minlength=76)
    hold_classes = np.bincount(JOINT_TARGET[hold_idx], minlength=76)

    if np.any(fit_classes == 0) or np.any(hold_classes == 0):
        raise RuntimeError(
            f"Fold {fold} is missing a joint class. "
            "Do not train until this is investigated."
        )

    FOLD_ASSIGNMENTS[hold_idx] = fold

    row = {
        "Fold": fold,
        "Fit_Windows": len(fit_idx),
        "Holdout_Windows": len(hold_idx),
        "Fit_Groups": len(fit_groups),
        "Holdout_Groups": len(hold_groups),
        "Joint_Classes_in_Holdout": int((hold_classes > 0).sum()),
        "Holdout_Event_Prevalence": float(
            (JOINT_TARGET[hold_idx] > 0).mean()
        )
    }

    for family_id, family_name in ID_TO_FAMILY.items():
        row[family_name] = int(
            np.sum(FAMILY[hold_idx] == family_id)
        )

    fold_rows.append(row)

assert np.all(FOLD_ASSIGNMENTS > 0)

# Every episode and calibration group must belong to one fold.
assert (
    pd.DataFrame({
        "episode": EPISODE_ID,
        "fold": FOLD_ASSIGNMENTS
    }).groupby("episode")["fold"].nunique() == 1
).all()

assert (
    pd.DataFrame({
        "group": CV_GROUPS,
        "fold": FOLD_ASSIGNMENTS
    }).groupby("group")["fold"].nunique() == 1
).all()

CV_SUMMARY = pd.DataFrame(fold_rows)

print("=" * 90)
print("COMPETING-RISK CROSS-VALIDATION VERIFIED")
print("=" * 90)
print(CV_SUMMARY.round(6).to_string(index=False))
print("\nCalibration-group overlap: ZERO")
print("Episode overlap: ZERO")
print("All 76 classes present in every holdout: PASS")
print("Official VALIDATION/TEST: NOT LOADED")
print("Training: NOT STARTED")


In [ ]:

# =============================================================================
# LSTM V2 COMPETING-RISK — CELL 4
# TEMPORAL ROBUST PREPROCESSING
#
# NO TRAINING; NO VALIDATION/TEST ACCESS.
# =============================================================================

class TemporalRobustScaler:

    def __init__(self, clip_value=8.0):
        self.clip_value = float(clip_value)
        self.median_ = None
        self.scale_ = None

    def fit(self, X):
        assert X.ndim == 3 and X.shape[1:] == (10, 14)
        assert np.isfinite(X).all()
        assert np.all(X >= 0)

        logged = np.log1p(X.astype(np.float32))
        flattened = logged.reshape(-1, 14)

        q25, q50, q75 = np.quantile(
            flattened, [0.25, 0.50, 0.75], axis=0
        )

        self.median_ = q50.astype(np.float32)
        self.scale_ = np.maximum(
            (q75 - q25).astype(np.float32), 0.05
        )
        return self

    def transform(self, X):
        if self.median_ is None:
            raise RuntimeError("Scaler must be fitted first.")

        logged = np.log1p(X.astype(np.float32))
        scaled = (
            logged - self.median_[None, None, :]
        ) / self.scale_[None, None, :]

        return np.ascontiguousarray(
            np.clip(scaled, -self.clip_value, self.clip_value),
            dtype=np.float32
        )

    def state_dict(self):
        if self.median_ is None:
            raise RuntimeError("Scaler has not been fitted.")

        return {
            "median": self.median_.tolist(),
            "scale": self.scale_.tolist(),
            "clip_value": self.clip_value,
            "feature_names": FEATURES
        }


# Test on a fitting partition from fold 1.
# Do not use the fold holdout to estimate scaling parameters.
preview_fit_idx, preview_hold_idx = CV_FOLDS[0]

PREVIEW_SCALER = TemporalRobustScaler().fit(
    X_RAW[preview_fit_idx]
)

X_PREVIEW_FIT = PREVIEW_SCALER.transform(
    X_RAW[preview_fit_idx[:64]]
)

X_PREVIEW_HOLD = PREVIEW_SCALER.transform(
    X_RAW[preview_hold_idx[:64]]
)

assert X_PREVIEW_FIT.shape == (64, 10, 14)
assert X_PREVIEW_HOLD.shape == (64, 10, 14)
assert np.isfinite(X_PREVIEW_FIT).all()
assert np.isfinite(X_PREVIEW_HOLD).all()

print("=" * 90)
print("COMPETING-RISK PREPROCESSING VERIFIED")
print("=" * 90)
print("Method: log1p + median/IQR")
print("Clipping: [-8, 8]")
print("Scaler fitted on: Fold 1 fitting partition ONLY")
print("Fit preview:", X_PREVIEW_FIT.shape)
print("Holdout preview:", X_PREVIEW_HOLD.shape)
print("Future CV: separate scaler per fold")
print("Future full-TRAIN fit: separate full-TRAIN scaler")
print("Training: NOT STARTED")


In [ ]:

# =============================================================================
# LSTM V2 COMPETING-RISK — CELL 5
# DEFINE THE JOINT ONSET-MINUTE × ATTACK-FAMILY MODEL
#
# NO TRAINING
# =============================================================================

import torch.nn as nn
import torch.nn.functional as F


class LSTMCompetingRisk(nn.Module):

    def __init__(
        self,
        input_dim=14,
        hidden_dim=64,
        num_layers=2,
        dropout=0.10,
        dense_dim=32,
        num_classes=76
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.projection = nn.Sequential(
            nn.Linear(hidden_dim, dense_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classifier = nn.Linear(
            dense_dim,
            num_classes
        )

    def forward(self, x):
        if x.ndim != 3 or x.shape[1:] != (10, 14):
            raise ValueError(
                f"Expected (batch,10,14), received {tuple(x.shape)}"
            )

        _, (hidden, _) = self.lstm(x)

        representation = self.projection(hidden[-1])
        logits = self.classifier(representation)

        return logits


CR_MODEL_CONFIG = {
    "input_dim": 14,
    "hidden_dim": 64,
    "num_layers": 2,
    "dropout": 0.10,
    "dense_dim": 32,
    "num_classes": 76
}

CR_MODEL = LSTMCompetingRisk(**CR_MODEL_CONFIG).to(DEVICE)

parameter_count = sum(
    parameter.numel()
    for parameter in CR_MODEL.parameters()
)

print("=" * 90)
print("COMPETING-RISK LSTM ARCHITECTURE")
print("=" * 90)
print(CR_MODEL)
print("\nTrainable parameters:", parameter_count)
print("Input: (batch, 10, 14)")
print("Output: (batch, 76) raw logits")
print("Loss for later training: CrossEntropyLoss")
print("Weights: RANDOMLY INITIALIZED")
print("Training: NOT STARTED")


In [ ]:

# =============================================================================
# LSTM V2 COMPETING-RISK — CELL 6
# JOINT DISTRIBUTION + FORECASTING SMOKE TEST
#
# NO OPTIMIZER / BACKWARD PASS / TRAINING.
# =============================================================================

def competing_risk_outputs(logits):
    """
    Input:
        logits: [batch, 76]

    Returns:
        joint_probs: [batch, 76]
        event_matrix: [batch, 15, 5]
        horizon_probs: [batch, 3]   (H5,H10,H15)
        family_probs: [batch, 5]    (joint with onset <=15)
        onset_probs: [batch, 15]    (joint with onset <=15)
        no_event_prob: [batch]
        conditional_family_probs: [batch, 5]
        conditional_onset_probs: [batch, 15]

    Conditional outputs are zero when the event probability
    is numerically negligible. Such rows have no meaningful
    conditional family or ETA forecast.
    """

    if logits.ndim != 2 or logits.shape[1] != 76:
        raise ValueError("Expected logits shape [batch,76]")

    joint_probs = torch.softmax(logits, dim=1)

    no_event = joint_probs[:, 0]
    event_matrix = joint_probs[:, 1:].reshape(-1, 15, 5)

    # Marginal probabilities by onset minute and family.
    onset_probs = event_matrix.sum(dim=2)
    family_probs = event_matrix.sum(dim=1)

    # Cumulative event probability within each horizon.
    cumulative = onset_probs.cumsum(dim=1)
    horizon_probs = cumulative[:, [4, 9, 14]]

    event_15 = horizon_probs[:, 2]

    safe_denom = event_15.clamp_min(1e-12)

    conditional_family = family_probs / safe_denom.unsqueeze(1)
    conditional_onset = onset_probs / safe_denom.unsqueeze(1)

    valid_condition = event_15 > 1e-10

    conditional_family = torch.where(
        valid_condition.unsqueeze(1),
        conditional_family,
        torch.zeros_like(conditional_family)
    )

    conditional_onset = torch.where(
        valid_condition.unsqueeze(1),
        conditional_onset,
        torch.zeros_like(conditional_onset)
    )

    return {
        "joint_probs": joint_probs,
        "event_matrix": event_matrix,
        "horizon_probs": horizon_probs,
        "family_probs": family_probs,
        "onset_probs": onset_probs,
        "no_event_prob": no_event,
        "conditional_family_probs": conditional_family,
        "conditional_onset_probs": conditional_onset
    }


def expected_eta_if_event(outputs):
    """
    Conditional expected onset minute, given an attack onset
    within the 15-minute forecasting horizon.
    """

    onset_conditional = outputs["conditional_onset_probs"]
    minutes = torch.arange(
        1, 16,
        device=onset_conditional.device,
        dtype=onset_conditional.dtype
    )

    eta = (onset_conditional * minutes).sum(dim=1)

    valid = outputs["horizon_probs"][:, 2] > 1e-10

    return torch.where(
        valid,
        eta,
        torch.full_like(eta, float("nan"))
    )


# --------------------------------------------------------------------------
# A. Known-target encoding tests
# --------------------------------------------------------------------------

test_onsets = np.array([0, 1, 1, 7, 15, 15])
test_families = np.array([0, 1, 5, 3, 1, 5])

test_classes = np.where(
    test_onsets == 0,
    0,
    (test_onsets - 1) * 5 + test_families
)

assert test_classes.tolist() == [0, 1, 5, 33, 71, 75]

# --------------------------------------------------------------------------
# B. One-hot logits with nearly certain event outcomes.
# --------------------------------------------------------------------------

test_logits = torch.full((6, 76), -30.0)
test_logits[
    torch.arange(6),
    torch.tensor(test_classes, dtype=torch.long)
] = 30.0

known_outputs = competing_risk_outputs(test_logits)
known_horizons = known_outputs["horizon_probs"].cpu().numpy()

expected_horizons = np.array([
    [0, 0, 0],  # no event
    [1, 1, 1],  # onset 1
    [1, 1, 1],  # onset 1
    [0, 1, 1],  # onset 7
    [0, 0, 1],  # onset 15
    [0, 0, 1],  # onset 15
], dtype=np.float32)

assert np.allclose(
    known_horizons,
    expected_horizons,
    atol=1e-6
)

# --------------------------------------------------------------------------
# C. Untrained model forward pass on an actual TRAIN preview.
# --------------------------------------------------------------------------

CR_MODEL.eval()

preview_x = torch.from_numpy(
    X_PREVIEW_FIT[:32]
).to(DEVICE)

preview_y = torch.from_numpy(
    JOINT_TARGET[preview_fit_idx[:32]]
).long().to(DEVICE)

with torch.inference_mode():
    logits = CR_MODEL(preview_x)
    output = competing_risk_outputs(logits)

    assert logits.shape == (32, 76)

    joint = output["joint_probs"]
    matrix = output["event_matrix"]
    horizon = output["horizon_probs"]
    family_prob = output["family_probs"]
    onset_prob = output["onset_probs"]
    no_event = output["no_event_prob"]

    assert matrix.shape == (32, 15, 5)
    assert horizon.shape == (32, 3)
    assert family_prob.shape == (32, 5)
    assert onset_prob.shape == (32, 15)

    assert torch.isfinite(joint).all()
    assert (joint >= 0).all()
    assert (joint <= 1).all()

    assert torch.allclose(
        joint.sum(dim=1),
        torch.ones(32, device=DEVICE),
        atol=1e-6
    )

    assert torch.allclose(
        no_event + horizon[:, 2],
        torch.ones(32, device=DEVICE),
        atol=1e-6
    )

    assert torch.allclose(
        family_prob.sum(dim=1),
        horizon[:, 2],
        atol=1e-6
    )

    assert torch.allclose(
        onset_prob.sum(dim=1),
        horizon[:, 2],
        atol=1e-6
    )

    assert torch.all(horizon[:, 0] <= horizon[:, 1] + 1e-7)
    assert torch.all(horizon[:, 1] <= horizon[:, 2] + 1e-7)

    conditional_family = output["conditional_family_probs"]
    conditional_onset = output["conditional_onset_probs"]

    assert torch.allclose(
        conditional_family.sum(dim=1),
        torch.ones(32, device=DEVICE),
        atol=1e-5
    )

    assert torch.allclose(
        conditional_onset.sum(dim=1),
        torch.ones(32, device=DEVICE),
        atol=1e-5
    )

    # Cross-entropy is a 76-outcome likelihood.
    preview_loss = F.cross_entropy(logits, preview_y)
    assert torch.isfinite(preview_loss)

    eta = expected_eta_if_event(output)
    assert torch.isfinite(eta).all()
    assert torch.all((eta >= 1) & (eta <= 15))

print("=" * 90)
print("COMPETING-RISK LSTM — SMOKE TEST COMPLETE")
print("=" * 90)
print("Known target mapping: PASS")
print("76-class probabilities sum to 1: PASS")
print("Horizon probability ordering: PASS")
print("Family probability marginalization: PASS")
print("Onset probability marginalization: PASS")
print("No-event complement: PASS")
print("Conditional probabilities: PASS")
print("Cross-entropy finite: PASS")
print("Conditional expected ETA: PASS")

print("\nUntrained loss:", round(float(preview_loss.item()), 6))
print("Output tensor shape:", tuple(logits.shape))

print("\nIllustrative untrained predictions (first 3):")
for i in range(3):
    p = horizon[i].cpu().numpy()
    family_idx = int(conditional_family[i].argmax().item())

    print(
        f"Example {i + 1}: "
        f"H5={p[0]:.4f}, "
        f"H10={p[1]:.4f}, "
        f"H15={p[2]:.4f}, "
        f"Top conditional family={FAMILIES[family_idx]}, "
        f"Expected ETA={eta[i].item():.2f} min"
    )

print("\nModel training: NOT STARTED")
print("Optimizer steps: ZERO")
print("Official VALIDATION/TEST: NOT LOADED")
print("Frozen dataset: UNCHANGED")


In [ ]:

# =============================================================================
# LSTM V2 COMPETING-RISK — CELL 7
# CONTROLLED TRAIN-ONLY SANITY FIT
#
# WARNING: EXECUTING THIS CELL STARTS TRAINING.
# Uses GroupKFold fold 1 only.
# No official VALIDATION or TEST access.
# No final model fitting or hyperparameter selection.
# =============================================================================

import time
import random
import numpy as np
import torch
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import average_precision_score

CR_SANITY_SEED = 20261024
CR_SANITY_EPOCHS = 8
CR_SANITY_BATCH_SIZE = 256

random.seed(CR_SANITY_SEED)
np.random.seed(CR_SANITY_SEED)
torch.manual_seed(CR_SANITY_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CR_SANITY_SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

CR_SANITY_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 100)
print("COMPETING-RISK LSTM — FOLD 1 DIAGNOSTIC")
print("=" * 100)
print("Device:", CR_SANITY_DEVICE)
print("Epochs:", CR_SANITY_EPOCHS)
print("Training: WILL START IN THIS CELL")
print("Official VALIDATION/TEST: NOT ACCESSED")

fit_idx, hold_idx = CV_FOLDS[0]

assert set(CV_GROUPS[fit_idx]).isdisjoint(
    set(CV_GROUPS[hold_idx])
)

assert set(EPISODE_ID[fit_idx]).isdisjoint(
    set(EPISODE_ID[hold_idx])
)

# Fit a fresh scaler using ONLY fold 1 fitting windows.
sanity_scaler = TemporalRobustScaler().fit(
    X_RAW[fit_idx]
)

X_fit = sanity_scaler.transform(X_RAW[fit_idx])
X_hold = sanity_scaler.transform(X_RAW[hold_idx])

y_fit = JOINT_TARGET[fit_idx].astype(np.int64)
y_hold = JOINT_TARGET[hold_idx].astype(np.int64)

train_data = TensorDataset(
    torch.from_numpy(X_fit),
    torch.from_numpy(y_fit)
)

hold_data = TensorDataset(
    torch.from_numpy(X_hold),
    torch.from_numpy(y_hold)
)

generator = torch.Generator().manual_seed(CR_SANITY_SEED)

train_loader = DataLoader(
    train_data,
    batch_size=CR_SANITY_BATCH_SIZE,
    shuffle=True,
    generator=generator,
    num_workers=0,
    pin_memory=CR_SANITY_DEVICE.type == "cuda"
)

hold_loader = DataLoader(
    hold_data,
    batch_size=1024,
    shuffle=False,
    num_workers=0,
    pin_memory=CR_SANITY_DEVICE.type == "cuda"
)

# Fresh model; never train the earlier smoke-test instance.
sanity_model = LSTMCompetingRisk(
    **CR_MODEL_CONFIG
).to(CR_SANITY_DEVICE)

optimizer = torch.optim.AdamW(
    sanity_model.parameters(),
    lr=0.001,
    weight_decay=0.001
)

history = []
start = time.perf_counter()

for epoch in range(1, CR_SANITY_EPOCHS + 1):

    sanity_model.train()
    loss_sum = 0.0
    count = 0

    for xb, yb in train_loader:

        xb = xb.to(CR_SANITY_DEVICE)
        yb = yb.to(CR_SANITY_DEVICE)

        optimizer.zero_grad(set_to_none=True)

        logits = sanity_model(xb)
        loss = F.cross_entropy(logits, yb)

        if not torch.isfinite(loss):
            raise RuntimeError("Nonfinite training loss.")

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            sanity_model.parameters(),
            max_norm=1.0,
            error_if_nonfinite=True
        )

        optimizer.step()

        loss_sum += float(loss.item()) * len(xb)
        count += len(xb)

    train_nll = loss_sum / count

    # Holdout is part of TRAIN CV, not official VALIDATION.
    sanity_model.eval()

    logits_parts = []
    hold_loss_sum = 0
    hold_count = 0

    with torch.inference_mode():

        for xb, yb in hold_loader:

            xb = xb.to(CR_SANITY_DEVICE)
            yb = yb.to(CR_SANITY_DEVICE)

            logits = sanity_model(xb)

            hold_loss_sum += float(
                F.cross_entropy(logits, yb, reduction="sum").item()
            )
            hold_count += len(yb)

            logits_parts.append(logits.cpu())

    hold_nll = hold_loss_sum / hold_count

    logits_all = torch.cat(logits_parts)
    outputs = competing_risk_outputs(logits_all)

    horizon_p = outputs["horizon_probs"].numpy()
    family_cond = outputs["conditional_family_probs"].numpy()
    onset_cond = outputs["conditional_onset_probs"].numpy()

    hold_horizons = Y_HORIZONS[hold_idx]

    ap = [
        float(average_precision_score(
            hold_horizons[:, j],
            horizon_p[:, j]
        ))
        for j in range(3)
    ]

    # Family and ETA are evaluated on actual event-positive windows.
    # These are diagnostic conditional metrics, not deployment metrics.
    event_mask = ONSET[hold_idx] > 0

    true_families = FAMILY[hold_idx][event_mask]
    predicted_families = (
        family_cond[event_mask].argmax(axis=1) + 1
    )

    family_accuracy = float(
        np.mean(predicted_families == true_families)
    )

    minute_grid = np.arange(1, 16, dtype=np.float64)
    expected_eta = onset_cond @ minute_grid

    eta_mae = float(
        np.mean(
            np.abs(
                expected_eta[event_mask]
                - ONSET[hold_idx][event_mask]
            )
        )
    )

    result = {
        "epoch": epoch,
        "train_nll": train_nll,
        "hold_nll": hold_nll,
        "H5_AP": ap[0],
        "H10_AP": ap[1],
        "H15_AP": ap[2],
        "event_conditional_family_accuracy": family_accuracy,
        "event_conditional_eta_mae": eta_mae
    }

    history.append(result)

    print(
        f"Epoch {epoch:02d}/{CR_SANITY_EPOCHS} | "
        f"Train NLL={train_nll:.5f} | "
        f"Hold NLL={hold_nll:.5f} | "
        f"H5 AP={ap[0]:.5f} | "
        f"H10 AP={ap[1]:.5f} | "
        f"H15 AP={ap[2]:.5f} | "
        f"Family Acc={family_accuracy:.4f} | "
        f"ETA MAE={eta_mae:.3f}m"
    )

SANITY_HISTORY = pd.DataFrame(history)

print("\n" + "=" * 100)
print("COMPETING-RISK LSTM — SANITY FIT COMPLETE")
print("=" * 100)

print(SANITY_HISTORY.round(6).to_string(index=False))
print("\nElapsed seconds:", round(time.perf_counter() - start, 2))
print("Official VALIDATION: NOT ACCESSED")
print("Official TEST: NOT ACCESSED")
print("Final model: NOT TRAINED")
print("Hyperparameter search: NOT STARTED")
print("Frozen dataset: UNCHANGED")


In [ ]:

# =============================================================================
# LSTM V2 COMPETING-RISK — CELL 8
# FIVE-FOLD TRAIN-ONLY MODEL SELECTION
#
# WARNING: EXECUTING THIS CELL STARTS MODEL TRAINING.
#
# Primary selection:
#   MINIMUM MEAN FOLD HOLDOUT JOINT NLL
#
# Checkpoint selection:
#   MINIMUM HOLDOUT JOINT NLL PER FOLD
#
# Secondary diagnostics:
#   H5/H10/H15 AP
#   Event-conditional family macro-F1
#   Event-conditional family accuracy
#   Event-conditional onset ETA MAE
#
# All candidate models are newly initialized.
#
# NO:
#   - Official VALIDATION access
#   - Official TEST access
#   - Final full-TRAIN fitting
#   - Frozen dataset modifications
# =============================================================================

import os
import gc
import json
import time
import random
import hashlib
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.metrics import (
    average_precision_score,
    f1_score,
    accuracy_score
)

from torch.utils.data import TensorDataset, DataLoader

# =============================================================================
# 8A — FIXED SEARCH CONFIGURATIONS
# =============================================================================

SEARCH_CONFIGS = [
    {
        "id": "C01",
        "hidden_dim": 64,
        "num_layers": 2,
        "dropout": 0.20,
        "dense_dim": 32,
        "learning_rate": 0.001
    },
    {
        "id": "C02",
        "hidden_dim": 32,
        "num_layers": 2,
        "dropout": 0.20,
        "dense_dim": 32,
        "learning_rate": 0.001
    },
    {
        "id": "C03",
        "hidden_dim": 128,
        "num_layers": 2,
        "dropout": 0.20,
        "dense_dim": 32,
        "learning_rate": 0.001
    },
    {
        "id": "C04",
        "hidden_dim": 64,
        "num_layers": 1,
        "dropout": 0.20,
        "dense_dim": 32,
        "learning_rate": 0.001
    },
    {
        "id": "C05",
        "hidden_dim": 64,
        "num_layers": 2,
        "dropout": 0.10,
        "dense_dim": 32,
        "learning_rate": 0.001
    },
    {
        "id": "C06",
        "hidden_dim": 64,
        "num_layers": 2,
        "dropout": 0.35,
        "dense_dim": 32,
        "learning_rate": 0.001
    },
    {
        "id": "C07",
        "hidden_dim": 64,
        "num_layers": 2,
        "dropout": 0.20,
        "dense_dim": 32,
        "learning_rate": 0.0003
    },
    {
        "id": "C08",
        "hidden_dim": 64,
        "num_layers": 2,
        "dropout": 0.20,
        "dense_dim": 32,
        "learning_rate": 0.002
    }
]

SEARCH_SEED = 20261024

MAX_EPOCHS = 35
PATIENCE = 7
MIN_DELTA = 0.0001

BATCH_SIZE = 256
WEIGHT_DECAY = 0.001
GRAD_CLIP = 1.0

N_FOLDS = 5

CV_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CV_ROOT = (
    ROOT
    / "Synthetic_NAF_v2"
    / "models"
    / "LSTM_V2_CompetingRisk"
    / "train_cv_selection_GPU"
)

# =============================================================================
# 8B — VERIFY REQUIRED PREPARATION
# =============================================================================

required_names = [
    "X_RAW",
    "Y_HORIZONS",
    "JOINT_TARGET",
    "ONSET",
    "FAMILY",
    "EPISODE_ID",
    "CV_GROUPS",
    "CV_FOLDS",
    "FEATURES",
    "FAMILIES",
    "TemporalRobustScaler",
    "LSTMCompetingRisk",
    "competing_risk_outputs",
    "MANIFEST",
    "TRAIN_FILE"
]

missing = [
    name
    for name in required_names
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Please run preparation Cells 1–6 first.\n"
        f"Missing variables: {missing}"
    )

assert X_RAW.shape == (88186, 10, 14)
assert Y_HORIZONS.shape == (88186, 3)
assert JOINT_TARGET.shape == (88186,)
assert len(CV_FOLDS) == N_FOLDS

assert set(np.unique(JOINT_TARGET)) == set(range(76))

assert len(np.unique(EPISODE_ID)) == 1400
assert len(np.unique(CV_GROUPS)) == 35

# Verify original frozen TRAIN data again.
expected_train_hash = (
    MANIFEST["data_integrity"]["file_sha256"]
    ["windows_W10_TRAIN.npz"]
)

assert sha256_file(TRAIN_FILE) == expected_train_hash

for fold_id, (fit_idx, hold_idx) in enumerate(CV_FOLDS, 1):

    assert set(CV_GROUPS[fit_idx]).isdisjoint(
        set(CV_GROUPS[hold_idx])
    )

    assert set(EPISODE_ID[fit_idx]).isdisjoint(
        set(EPISODE_ID[hold_idx])
    )

    assert len(np.unique(CV_GROUPS[hold_idx])) == 7

    assert set(np.unique(JOINT_TARGET[hold_idx])) == set(
        range(76)
    )

if CV_DEVICE.type != "cuda":
    raise RuntimeError(
        "GPU experiment requires CUDA. "
        "Select a T4 GPU runtime and rerun Cells 1–6. "
        "No training has started."
    )

print("=" * 100)
print("COMPETING-RISK LSTM — FIVE-FOLD MODEL SELECTION")
print("=" * 100)

print("Device:", CV_DEVICE)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

print("Configurations:", len(SEARCH_CONFIGS))
print("Folds:", N_FOLDS)
print("Maximum epochs:", MAX_EPOCHS)
print("Early-stopping patience:", PATIENCE)
print("Batch size:", BATCH_SIZE)

print("\nPrimary selection: MINIMUM MEAN HOLDOUT JOINT NLL")
print("Checkpoint rule: MINIMUM FOLD HOLDOUT JOINT NLL")
print("Official VALIDATION: NOT ACCESSED")
print("Official TEST: NOT ACCESSED")

# =============================================================================
# 8C — REPRODUCIBILITY AND SAFE RESULT WRITING
# =============================================================================

def make_hash(obj):
    payload = json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


def atomic_json(path, obj):

    path = Path(path)

    temp_path = path.with_name(
        path.name + ".tmp"
    )

    temp_path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            allow_nan=False
        ),
        encoding="utf-8"
    )

    os.replace(temp_path, path)


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def seed_everything(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


# The protocol is immutable for resumability.
# Changing architecture, environment or search settings
# will require a new experiment directory.

PROTOCOL = {
    "experiment": "LSTM_V2_COMPETING_RISK_CV",
    "dataset": "Synthetic_NAF_v2",
    "release_version": "2.0.0",
    "train_file_sha256": expected_train_hash,
    "feature_names": list(FEATURES),
    "families": list(FAMILIES),
    "joint_target_classes": 76,
    "joint_encoding": "0_NO_EVENT_ELSE_5_TIMES_ONSET_MINUS_1_PLUS_FAMILY",
    "configurations": SEARCH_CONFIGS,
    "number_of_folds": N_FOLDS,
    "fold_strategy": "GROUP_K_FOLD_SOURCE_CALIBRATION_GROUP",
    "fold_holdout_groups": [
        sorted(set(CV_GROUPS[hold_idx].tolist()))
        for _, hold_idx in CV_FOLDS
    ],
    "fold_holdout_indices_sha256": [
        hashlib.sha256(
            np.asarray(hold_idx, dtype="<i8").tobytes()
        ).hexdigest()
        for _, hold_idx in CV_FOLDS
    ],
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": PATIENCE,
    "early_stopping_min_delta": MIN_DELTA,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "weight_decay": WEIGHT_DECAY,
    "gradient_clip_norm": GRAD_CLIP,
    "loss": "UNWEIGHTED_76_CLASS_CROSS_ENTROPY",
    "preprocessing": "LOG1P_MEDIAN_IQR_MIN_SCALE_0.05_CLIP_8",
    "checkpoint_rule": "MINIMUM_HOLDOUT_JOINT_NLL",
    "configuration_selection_rule": "MINIMUM_MEAN_FOLD_JOINT_NLL",
    "epoch_budget_rule": "MEDIAN_FOLD_SELECTED_EPOCH",
    "secondary_metrics": [
        "H5_AP",
        "H10_AP",
        "H15_AP",
        "FAMILY_MACRO_F1_CONDITIONAL_ON_TRUE_EVENT",
        "FAMILY_ACCURACY_CONDITIONAL_ON_TRUE_EVENT",
        "ETA_MAE_CONDITIONAL_ON_TRUE_EVENT"
    ],
    "seed": SEARCH_SEED,
    "device": str(CV_DEVICE),
    "torch_version": torch.__version__
}

PROTOCOL_HASH = make_hash(PROTOCOL)

PROTOCOL_FILE = CV_ROOT / "SEARCH_PROTOCOL.json"

CV_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

if PROTOCOL_FILE.exists():

    saved_protocol = read_json(PROTOCOL_FILE)

    if saved_protocol != PROTOCOL:
        raise RuntimeError(
            "Existing competing-risk CV results use a "
            "different protocol. Do not mix experiments.\n"
            "Inspect SEARCH_PROTOCOL.json or use a "
            "new experiment directory."
        )

else:

    existing_results = list(
        CV_ROOT.glob("C*/FOLD_*_RESULT.json")
    )

    if existing_results:
        raise RuntimeError(
            "Found fold results without a protocol file. "
            "Refusing unsafe resume."
        )

    atomic_json(
        PROTOCOL_FILE,
        PROTOCOL
    )

print("\n✓ Frozen TRAIN data verified")
print("✓ Five group-disjoint folds verified")
print("✓ Search protocol:", PROTOCOL_HASH[:20])
print("Results directory:", CV_ROOT)

# =============================================================================
# 8D — FOLD-LOCAL SCALERS
#
# These are never fitted on fold holdout data.
# =============================================================================

FOLD_DATA = []

for fold_id, (fit_idx, hold_idx) in enumerate(
    CV_FOLDS, start=1
):

    scaler = TemporalRobustScaler().fit(
        X_RAW[fit_idx]
    )

    x_fit = scaler.transform(
        X_RAW[fit_idx]
    )

    x_hold = scaler.transform(
        X_RAW[hold_idx]
    )

    y_fit = np.ascontiguousarray(
        JOINT_TARGET[fit_idx],
        dtype=np.int64
    )

    y_hold = np.ascontiguousarray(
        JOINT_TARGET[hold_idx],
        dtype=np.int64
    )

    assert np.isfinite(x_fit).all()
    assert np.isfinite(x_hold).all()

    FOLD_DATA.append({
        "fold": fold_id,
        "x_fit": x_fit,
        "x_hold": x_hold,
        "y_fit": y_fit,
        "y_hold": y_hold,
        "horizon_hold": Y_HORIZONS[hold_idx],
        "onset_hold": ONSET[hold_idx],
        "family_hold": FAMILY[hold_idx],
        "scaler_state": scaler.state_dict()
    })

print("✓ Five fold-local scalers fitted")

# =============================================================================
# 8E — EVALUATION METRICS
# =============================================================================

def evaluate_cr_predictions(
    logits_numpy,
    horizon_truth,
    onset_truth,
    family_truth
):

    logits_t = torch.from_numpy(
        logits_numpy.astype(np.float32)
    )

    with torch.inference_mode():
        outputs = competing_risk_outputs(logits_t)

    horizon_probs = (
        outputs["horizon_probs"]
        .numpy()
        .astype(np.float64)
    )

    family_cond = (
        outputs["conditional_family_probs"]
        .numpy()
        .astype(np.float64)
    )

    onset_cond = (
        outputs["conditional_onset_probs"]
        .numpy()
        .astype(np.float64)
    )

    ap_values = [
        float(
            average_precision_score(
                horizon_truth[:, col],
                horizon_probs[:, col]
            )
        )
        for col in range(3)
    ]

    event_mask = onset_truth > 0

    if not np.any(event_mask):
        raise RuntimeError(
            "No event-positive samples in fold holdout."
        )

    true_family = family_truth[event_mask]

    predicted_family = (
        np.argmax(
            family_cond[event_mask],
            axis=1
        ) + 1
    )

    macro_f1 = float(
        f1_score(
            true_family,
            predicted_family,
            labels=[1, 2, 3, 4, 5],
            average="macro",
            zero_division=0
        )
    )

    family_accuracy = float(
        accuracy_score(
            true_family,
            predicted_family
        )
    )

    minutes = np.arange(
        1, 16,
        dtype=np.float64
    )

    expected_eta = onset_cond @ minutes

    eta_mae = float(
        np.mean(
            np.abs(
                expected_eta[event_mask]
                - onset_truth[event_mask]
            )
        )
    )

    if not np.isfinite(eta_mae):
        raise RuntimeError("Nonfinite ETA MAE.")

    if np.any(
        horizon_probs[:, 0]
        > horizon_probs[:, 1] + 1e-7
    ):
        raise RuntimeError(
            "H5 > H10 probability violation."
        )

    if np.any(
        horizon_probs[:, 1]
        > horizon_probs[:, 2] + 1e-7
    ):
        raise RuntimeError(
            "H10 > H15 probability violation."
        )

    return {
        "H5_AP": ap_values[0],
        "H10_AP": ap_values[1],
        "H15_AP": ap_values[2],
        "Mean_Horizon_AP": float(
            np.mean(ap_values)
        ),
        "Family_Macro_F1": macro_f1,
        "Family_Accuracy": family_accuracy,
        "ETA_MAE": eta_mae
    }

# =============================================================================
# 8F — TRAIN ONE CONFIGURATION ON ONE FOLD
# =============================================================================

def fit_configuration_fold(
    config,
    config_index,
    fold_data
):

    fold_id = fold_data["fold"]

    local_seed = (
        SEARCH_SEED
        + config_index * 100
        + fold_id
    )

    seed_everything(local_seed)

    model_config = {
        "input_dim": 14,
        "hidden_dim": config["hidden_dim"],
        "num_layers": config["num_layers"],
        "dropout": config["dropout"],
        "dense_dim": config["dense_dim"],
        "num_classes": 76
    }

    model = LSTMCompetingRisk(
        **model_config
    ).to(CV_DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=WEIGHT_DECAY
    )

    fit_dataset = TensorDataset(
        torch.from_numpy(fold_data["x_fit"]),
        torch.from_numpy(fold_data["y_fit"])
    )

    hold_dataset = TensorDataset(
        torch.from_numpy(fold_data["x_hold"]),
        torch.from_numpy(fold_data["y_hold"])
    )

    generator = torch.Generator()
    generator.manual_seed(local_seed)

    fit_loader = DataLoader(
        fit_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=True
    )

    hold_loader = DataLoader(
        hold_dataset,
        batch_size=1024,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    best_nll = float("inf")
    best_epoch = None
    best_metrics = None
    best_actual_nll = None

    epochs_without_improvement = 0

    epoch_history = []

    start = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):

        # -----------------------------------------------------
        # TRAIN
        # -----------------------------------------------------

        model.train()

        train_loss_sum = 0.0
        train_seen = 0

        for xb, yb in fit_loader:

            xb = xb.to(
                CV_DEVICE,
                non_blocking=True
            )

            yb = yb.to(
                CV_DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(xb)

            loss = F.cross_entropy(
                logits,
                yb
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Nonfinite training loss: "
                    f"{config['id']} fold {fold_id}"
                )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=GRAD_CLIP,
                error_if_nonfinite=True
            )

            optimizer.step()

            batch_count = len(xb)

            train_loss_sum += (
                float(loss.item()) * batch_count
            )

            train_seen += batch_count

        train_nll = train_loss_sum / train_seen

        # -----------------------------------------------------
        # GROUP-DISJOINT HOLDOUT
        # -----------------------------------------------------

        model.eval()

        hold_loss_sum = 0.0
        hold_seen = 0

        logits_parts = []

        with torch.inference_mode():

            for xb, yb in hold_loader:

                xb = xb.to(
                    CV_DEVICE,
                    non_blocking=True
                )

                yb = yb.to(
                    CV_DEVICE,
                    non_blocking=True
                )

                logits = model(xb)

                batch_loss = F.cross_entropy(
                    logits,
                    yb,
                    reduction="sum"
                )

                hold_loss_sum += float(
                    batch_loss.item()
                )

                hold_seen += len(yb)

                logits_parts.append(
                    logits.cpu().numpy()
                )

        hold_nll = (
            hold_loss_sum / hold_seen
        )

        if not np.isfinite(hold_nll):
            raise RuntimeError(
                "Nonfinite holdout NLL."
            )

        all_logits = np.concatenate(
            logits_parts,
            axis=0
        )

        metrics = evaluate_cr_predictions(
            all_logits,
            fold_data["horizon_hold"],
            fold_data["onset_hold"],
            fold_data["family_hold"]
        )

        epoch_result = {
            "epoch": epoch,
            "train_nll": float(train_nll),
            "hold_nll": float(hold_nll),
            **metrics
        }

        epoch_history.append(
            epoch_result
        )

        print(
            f"{config['id']} | "
            f"Fold {fold_id} | "
            f"Epoch {epoch:02d} | "
            f"Train NLL={train_nll:.5f} | "
            f"Hold NLL={hold_nll:.5f} | "
            f"H5 AP={metrics['H5_AP']:.5f} | "
            f"H10 AP={metrics['H10_AP']:.5f} | "
            f"H15 AP={metrics['H15_AP']:.5f} | "
            f"Family F1={metrics['Family_Macro_F1']:.4f} | "
            f"ETA MAE={metrics['ETA_MAE']:.3f}"
        )

        # -----------------------------------------------------
        # NLL-BASED CHECKPOINT SELECTION
        #
        # We record metrics AT the selected epoch.
        # We do not independently maximize AP/F1/ETA.
        # -----------------------------------------------------

        if hold_nll < best_nll - MIN_DELTA:

            best_nll = hold_nll
            best_actual_nll = hold_nll
            best_epoch = epoch

            best_metrics = dict(metrics)

            epochs_without_improvement = 0

        else:

            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:

            print(
                f"  Early stopping: "
                f"best epoch {best_epoch}, "
                f"best holdout NLL {best_nll:.5f}"
            )

            break

    if best_epoch is None:
        raise RuntimeError(
            "No valid epoch selected."
        )

    elapsed = time.perf_counter() - start

    result = {
        "configuration_id": config["id"],
        "configuration": config,
        "fold": fold_id,
        "seed": local_seed,
        "protocol_hash": PROTOCOL_HASH,
        "fit_windows": int(
            len(fold_data["y_fit"])
        ),
        "holdout_windows": int(
            len(fold_data["y_hold"])
        ),
        "best_epoch": int(best_epoch),
        "best_holdout_joint_nll": float(
            best_actual_nll
        ),
        "selected_metrics": best_metrics,
        "epochs_run": len(epoch_history),
        "history": epoch_history,
        "elapsed_seconds": float(elapsed),
        "official_validation_used": False,
        "official_test_used": False
    }

    del model
    del optimizer
    del fit_loader
    del hold_loader

    gc.collect()
    torch.cuda.empty_cache()

    return result

# =============================================================================
# 8G — RUN OR RESUME ALL 40 FOLD EXPERIMENTS
# =============================================================================

ALL_FOLD_RESULTS = []

for config_index, config in enumerate(
    SEARCH_CONFIGS,
    start=1
):

    config_id = config["id"]

    config_dir = CV_ROOT / config_id

    config_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    print("\n" + "=" * 100)
    print(
        f"CONFIGURATION {config_index}/8: {config_id}"
    )
    print("=" * 100)
    print(config)

    for fold_data in FOLD_DATA:

        fold_id = fold_data["fold"]

        result_path = (
            config_dir
            / f"FOLD_{fold_id:02d}_RESULT.json"
        )

        if result_path.exists():

            result = read_json(result_path)

            if result["protocol_hash"] != PROTOCOL_HASH:
                raise RuntimeError(
                    f"Protocol mismatch: {result_path}"
                )

            if result["configuration_id"] != config_id:
                raise RuntimeError(
                    f"Configuration mismatch: {result_path}"
                )

            if result["fold"] != fold_id:
                raise RuntimeError(
                    f"Fold mismatch: {result_path}"
                )

            print(
                f"✓ Resumed saved {config_id} "
                f"fold {fold_id}; skipped retraining"
            )

        else:

            result = fit_configuration_fold(
                config,
                config_index,
                fold_data
            )

            atomic_json(
                result_path,
                result
            )

            print(
                f"✓ {config_id} fold {fold_id} "
                f"completed and saved"
            )

        ALL_FOLD_RESULTS.append(result)

assert len(ALL_FOLD_RESULTS) == 40

# =============================================================================
# 8H — BUILD CROSS-VALIDATION SUMMARY
# =============================================================================

fold_table_rows = []

for result in ALL_FOLD_RESULTS:

    row = {
        "Config_ID": result["configuration_id"],
        "Fold": result["fold"],
        "Best_Epoch": result["best_epoch"],
        "Holdout_Joint_NLL": (
            result["best_holdout_joint_nll"]
        ),
        **result["selected_metrics"]
    }

    fold_table_rows.append(row)

FOLD_RESULTS_DF = pd.DataFrame(
    fold_table_rows
)

summary_rows = []

for config in SEARCH_CONFIGS:

    config_id = config["id"]

    subset = FOLD_RESULTS_DF.loc[
        FOLD_RESULTS_DF["Config_ID"] == config_id
    ].sort_values("Fold")

    assert len(subset) == 5
    assert subset["Fold"].tolist() == [1, 2, 3, 4, 5]

    selected_epochs = subset[
        "Best_Epoch"
    ].to_numpy()

    final_epochs = int(
        np.median(selected_epochs)
    )

    summary_rows.append({
        "Config_ID": config_id,
        "Mean_Joint_NLL": float(
            subset["Holdout_Joint_NLL"].mean()
        ),
        "SD_Joint_NLL": float(
            subset["Holdout_Joint_NLL"].std(ddof=1)
        ),
        "Mean_H5_AP": float(
            subset["H5_AP"].mean()
        ),
        "Mean_H10_AP": float(
            subset["H10_AP"].mean()
        ),
        "Mean_H15_AP": float(
            subset["H15_AP"].mean()
        ),
        "Mean_Family_Macro_F1": float(
            subset["Family_Macro_F1"].mean()
        ),
        "Mean_Family_Accuracy": float(
            subset["Family_Accuracy"].mean()
        ),
        "Mean_ETA_MAE": float(
            subset["ETA_MAE"].mean()
        ),
        "Final_Epochs": final_epochs
    })

CV_SUMMARY_DF = pd.DataFrame(
    summary_rows
)

# Primary: lowest joint NLL.
# Ties: best family macro-F1, then config ID.
CV_SUMMARY_DF = (
    CV_SUMMARY_DF.sort_values(
        by=[
            "Mean_Joint_NLL",
            "Mean_Family_Macro_F1",
            "Config_ID"
        ],
        ascending=[
            True,
            False,
            True
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

# =============================================================================
# 8I — LOCK SELECTED CONFIGURATION
# =============================================================================

winner = CV_SUMMARY_DF.iloc[0]

winner_id = str(
    winner["Config_ID"]
)

winner_config = next(
    config
    for config in SEARCH_CONFIGS
    if config["id"] == winner_id
)

selection_record = {
    "dataset": "Synthetic_NAF_v2",
    "release_version": "2.0.0",
    "model_family": "LSTM_COMPETING_RISK_76_CLASS",
    "protocol_hash": PROTOCOL_HASH,
    "selection_rule": "MINIMUM_MEAN_HOLDOUT_JOINT_NLL",
    "checkpoint_rule": "MINIMUM_FOLD_HOLDOUT_JOINT_NLL",
    "selected_config_id": winner_id,
    "selected_model_config": winner_config,
    "selected_final_epochs": int(
        winner["Final_Epochs"]
    ),
    "selected_mean_joint_nll": float(
        winner["Mean_Joint_NLL"]
    ),
    "selected_mean_H5_AP": float(
        winner["Mean_H5_AP"]
    ),
    "selected_mean_H10_AP": float(
        winner["Mean_H10_AP"]
    ),
    "selected_mean_H15_AP": float(
        winner["Mean_H15_AP"]
    ),
    "selected_mean_family_macro_f1": float(
        winner["Mean_Family_Macro_F1"]
    ),
    "selected_mean_family_accuracy": float(
        winner["Mean_Family_Accuracy"]
    ),
    "selected_mean_eta_mae": float(
        winner["Mean_ETA_MAE"]
    ),
    "final_training_performed": False,
    "official_validation_used": False,
    "official_test_used": False
}

selection_file = (
    CV_ROOT
    / "SELECTED_CR_CONFIGURATION.json"
)

if selection_file.exists():

    prior = read_json(selection_file)

    # Do not silently change an already locked winner.
    stable_fields = [
        "protocol_hash",
        "selected_config_id",
        "selected_model_config",
        "selected_final_epochs"
    ]

    for field in stable_fields:
        if prior[field] != selection_record[field]:
            raise RuntimeError(
                "Existing model selection differs from "
                "the recomputed winner. Refusing overwrite."
            )

else:

    selection_record["created_utc"] = (
        datetime.now(timezone.utc).isoformat()
    )

    atomic_json(
        selection_file,
        selection_record
    )

# Save summary tables without overwriting different content.
summary_files = {
    "ALL_CONFIGURATIONS_CV_SUMMARY.csv": CV_SUMMARY_DF,
    "ALL_FOLD_RESULTS.csv": FOLD_RESULTS_DF
}

for filename, dataframe in summary_files.items():

    output_path = CV_ROOT / filename

    csv_text = dataframe.to_csv(
        index=False,
        float_format="%.12g"
    )

    if output_path.exists():

        old_text = output_path.read_text(
            encoding="utf-8"
        )

        if old_text != csv_text:
            raise RuntimeError(
                f"Existing summary differs: {output_path}"
            )

    else:

        output_path.write_text(
            csv_text,
            encoding="utf-8"
        )

# =============================================================================
# 8J — FINAL REPORT
# =============================================================================

print("\n" + "=" * 110)
print("COMPETING-RISK LSTM — FIVE-FOLD MODEL SELECTION COMPLETE")
print("=" * 110)

print("\nALL CONFIGURATION RESULTS")

print(
    CV_SUMMARY_DF.round(6).to_string(
        index=False
    )
)

print("\nSELECTED CONFIGURATION")
print(
    json.dumps(
        selection_record,
        indent=2
    )
)

print("\nCompleted fold experiments:", len(ALL_FOLD_RESULTS))
print("Results directory:", CV_ROOT)

print("\nFinal full-TRAIN competing-risk LSTM: NOT TRAINED")
print("Official VALIDATION: NOT ACCESSED")
print("Official TEST: NOT ACCESSED")
print("Frozen dataset: UNCHANGED")

print("\nNEXT: Review the joint NLL, horizon AP,")
print("family macro-F1 and ETA MAE before final fitting.")


In [ ]:

# ==========================================================================================
# LSTM V2 COMPETING-RISK — CELL 9
# FINAL FULL-TRAIN MODEL FITTING
#
# WARNING: EXECUTING THIS CELL STARTS TRAINING.
#
# Selected configuration: C01
# Epoch budget: 17, fixed from five-fold TRAIN CV
# Input: 10 minutes x 14 features
# Output: 76 joint classes (no onset + 15 minutes x 5 families)
#
# NO official VALIDATION / TEST access.
# NO additional model selection.
# NO changes to frozen dataset.
# NO overwriting an existing final model.
# ==========================================================================================

import os
import gc
import json
import time
import random
import hashlib
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader

# ==========================================================================================
# 9A — PATHS
# ==========================================================================================

ROOT_9 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

TRAIN_PATH_9 = (
    ROOT_9
    / "Synthetic_NAF_v2"
    / "release_candidate_corrected"
    / "windows_W10_TRAIN.npz"
)

FREEZE_DIR_9 = (
    ROOT_9
    / "Synthetic_NAF_v2"
    / "FINAL_RELEASE_FREEZE"
)

MANIFEST_PATH_9 = (
    FREEZE_DIR_9
    / "Synthetic_NAF_v2_RELEASE_MANIFEST.json"
)

LOCK_PATH_9 = (
    FREEZE_DIR_9
    / "Synthetic_NAF_v2_FREEZE_LOCK.json"
)

CV_ROOT_9 = (
    ROOT_9
    / "Synthetic_NAF_v2"
    / "models"
    / "LSTM_V2_CompetingRisk"
    / "train_cv_selection_GPU"
)

SELECTION_PATH_9 = (
    CV_ROOT_9
    / "SELECTED_CR_CONFIGURATION.json"
)

PROTOCOL_PATH_9 = (
    CV_ROOT_9
    / "SEARCH_PROTOCOL.json"
)

FINAL_DIR_9 = (
    ROOT_9
    / "Synthetic_NAF_v2"
    / "models"
    / "LSTM_V2_CompetingRisk"
    / "final_train_model"
)

# Protect any existing final release.
if FINAL_DIR_9.exists():
    raise RuntimeError(
        "Final model directory already exists:\n"
        f"{FINAL_DIR_9}\n"
        "Refusing to overwrite or replace an existing release."
    )

# ==========================================================================================
# 9B — REQUIRED PREPARATION VARIABLES
# ==========================================================================================

REQUIRED_9 = [
    "X_RAW",
    "Y_HORIZONS",
    "ONSET",
    "FAMILY",
    "JOINT_TARGET",
    "EPISODE_ID",
    "CV_GROUPS",
    "FEATURES",
    "FAMILIES",
    "TemporalRobustScaler",
    "LSTMCompetingRisk",
    "competing_risk_outputs"
]

missing_9 = [
    name for name in REQUIRED_9
    if name not in globals()
]

if missing_9:
    raise RuntimeError(
        "Run preparation Cells 1–6 before Cell 9.\n"
        f"Missing: {missing_9}"
    )

assert X_RAW.shape == (88186, 10, 14)
assert JOINT_TARGET.shape == (88186,)
assert Y_HORIZONS.shape == (88186, 3)

assert len(np.unique(EPISODE_ID)) == 1400
assert len(np.unique(CV_GROUPS)) == 35

assert len(FEATURES) == 14
assert len(FAMILIES) == 5

assert np.isfinite(X_RAW).all()
assert np.all(X_RAW >= 0)

assert np.all(
    (JOINT_TARGET >= 0) &
    (JOINT_TARGET <= 75)
)

assert set(np.unique(JOINT_TARGET)) == set(range(76))

expected_joint_9 = np.where(
    ONSET == 0,
    0,
    (ONSET - 1) * 5 + FAMILY
)

assert np.array_equal(
    expected_joint_9,
    JOINT_TARGET
)

for col, horizon in enumerate([5, 10, 15]):
    expected_y = (
        (ONSET >= 1) &
        (ONSET <= horizon)
    ).astype(np.int8)

    assert np.array_equal(
        Y_HORIZONS[:, col],
        expected_y
    )

# ==========================================================================================
# 9C — INTEGRITY HELPERS
# ==========================================================================================

def read_json_9(path):
    if not path.is_file():
        raise FileNotFoundError(path)

    return json.loads(
        path.read_text(encoding="utf-8")
    )


def file_sha256_9(path):
    digest = hashlib.sha256()

    with open(path, "rb") as stream:
        for chunk in iter(
            lambda: stream.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def canonical_sha256_9(obj):
    payload = json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


def write_json_9(path, obj):
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            allow_nan=False
        ),
        encoding="utf-8"
    )


# ==========================================================================================
# 9D — VERIFY FROZEN RELEASE AND SELECTED C01
# ==========================================================================================

manifest_9 = read_json_9(MANIFEST_PATH_9)
freeze_lock_9 = read_json_9(LOCK_PATH_9)

selection_9 = read_json_9(SELECTION_PATH_9)
protocol_9 = read_json_9(PROTOCOL_PATH_9)

assert manifest_9["release_version"] == "2.0.0"

assert (
    canonical_sha256_9(manifest_9)
    == freeze_lock_9["manifest_sha256_canonical"]
)

expected_train_hash_9 = (
    manifest_9["data_integrity"]["file_sha256"]
    ["windows_W10_TRAIN.npz"]
)

if file_sha256_9(TRAIN_PATH_9) != expected_train_hash_9:
    raise RuntimeError(
        "Frozen TRAIN dataset integrity failure."
    )

if (
    canonical_sha256_9(protocol_9)
    != selection_9["protocol_hash"]
):
    raise RuntimeError(
        "CV selection and protocol hashes do not match."
    )

assert selection_9["selected_config_id"] == "C01"
assert selection_9["selected_final_epochs"] == 17

assert selection_9["final_training_performed"] is False
assert selection_9["official_validation_used"] is False
assert selection_9["official_test_used"] is False

assert selection_9["selection_rule"] == (
    "MINIMUM_MEAN_HOLDOUT_JOINT_NLL"
)

assert protocol_9["loss"] == (
    "UNWEIGHTED_76_CLASS_CROSS_ENTROPY"
)

assert protocol_9["configuration_selection_rule"] == (
    "MINIMUM_MEAN_FOLD_JOINT_NLL"
)

assert protocol_9["number_of_folds"] == 5

selected_config_9 = selection_9["selected_model_config"]

EXPECTED_CONFIG_9 = {
    "id": "C01",
    "hidden_dim": 64,
    "num_layers": 2,
    "dropout": 0.20,
    "dense_dim": 32,
    "learning_rate": 0.001
}

if selected_config_9 != EXPECTED_CONFIG_9:
    raise RuntimeError(
        "Saved C01 hyperparameters differ from locked configuration."
    )

# Validate every saved fold result.
all_fold_results_9 = []

for config_number in range(1, 9):

    config_id = f"C{config_number:02d}"

    for fold_number in range(1, 6):

        path = (
            CV_ROOT_9
            / config_id
            / f"FOLD_{fold_number:02d}_RESULT.json"
        )

        result = read_json_9(path)

        assert result["protocol_hash"] == (
            selection_9["protocol_hash"]
        )

        assert result["configuration_id"] == config_id
        assert result["fold"] == fold_number

        assert result["official_validation_used"] is False
        assert result["official_test_used"] is False

        all_fold_results_9.append(result)

assert len(all_fold_results_9) == 40

# Independently recompute the selected winner and epoch budget.
ranking_9 = []

for config_id in [
    f"C{i:02d}" for i in range(1, 9)
]:

    subset = [
        r for r in all_fold_results_9
        if r["configuration_id"] == config_id
    ]

    assert len(subset) == 5

    ranking_9.append({
        "config": config_id,
        "mean_nll": float(
            np.mean([
                r["best_holdout_joint_nll"]
                for r in subset
            ])
        ),
        "mean_family_f1": float(
            np.mean([
                r["selected_metrics"]["Family_Macro_F1"]
                for r in subset
            ])
        ),
        "median_epoch": int(
            np.median([
                r["best_epoch"]
                for r in subset
            ])
        )
    })

ranking_9.sort(
    key=lambda r: (
        r["mean_nll"],
        -r["mean_family_f1"],
        r["config"]
    )
)

assert ranking_9[0]["config"] == "C01"
assert ranking_9[0]["median_epoch"] == 17

assert np.isclose(
    ranking_9[0]["mean_nll"],
    selection_9["selected_mean_joint_nll"],
    rtol=0,
    atol=1e-10
)

# Verify current notebook arrays against frozen TRAIN.
with np.load(
    TRAIN_PATH_9,
    allow_pickle=False
) as frozen:

    if not np.array_equal(
        X_RAW,
        frozen["X"]
    ):
        raise RuntimeError(
            "Notebook TRAIN features differ from frozen features."
        )

    if not np.array_equal(
        JOINT_TARGET,
        np.where(
            frozen["onset_class"] == 0,
            0,
            (frozen["onset_class"] - 1) * 5
            + frozen["family_class"]
        )
    ):
        raise RuntimeError(
            "Notebook joint labels differ from frozen labels."
        )

print("✓ Frozen dataset integrity verified")
print("✓ TRAIN arrays verified")
print("✓ All 40 CV fold records verified")
print("✓ C01 selection independently verified")
print("✓ Fixed 17-epoch training budget verified")

# ==========================================================================================
# 9E — LOCK TRAINING SETTINGS
# ==========================================================================================

FINAL_SEED_9 = 20261024

FINAL_EPOCHS_9 = 17
FINAL_BATCH_SIZE_9 = 256

FINAL_LR_9 = 0.001
FINAL_WEIGHT_DECAY_9 = 0.001

FINAL_GRAD_CLIP_9 = 1.0

FINAL_DEVICE_9 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if FINAL_DEVICE_9.type != "cuda":
    raise RuntimeError(
        "This final GPU experiment requires CUDA. "
        "Select T4 GPU and rerun the preparation cells."
    )

random.seed(FINAL_SEED_9)
np.random.seed(FINAL_SEED_9)
torch.manual_seed(FINAL_SEED_9)
torch.cuda.manual_seed_all(FINAL_SEED_9)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

print("\n" + "=" * 100)
print("COMPETING-RISK LSTM — FINAL FULL-TRAIN FITTING")
print("=" * 100)

print("Device:", FINAL_DEVICE_9)
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

print("\nSelected configuration:", "C01")
print("Final epochs:", FINAL_EPOCHS_9)
print("Batch size:", FINAL_BATCH_SIZE_9)
print("Learning rate:", FINAL_LR_9)
print("Weight decay:", FINAL_WEIGHT_DECAY_9)
print("Gradient clipping:", FINAL_GRAD_CLIP_9)

print("\nTRAIN windows:", len(X_RAW))
print("TRAIN episodes:", len(np.unique(EPISODE_ID)))
print("Joint output classes:", 76)

print("\nOfficial VALIDATION: NOT LOADED")
print("Official TEST: NOT LOADED")

# ==========================================================================================
# 9F — FIT A FRESH SCALER ON FULL TRAIN ONLY
# ==========================================================================================

FINAL_CR_SCALER = TemporalRobustScaler(
    clip_value=8.0
)

FINAL_CR_SCALER.fit(
    X_RAW
)

X_FULL_9 = FINAL_CR_SCALER.transform(
    X_RAW
)

Y_FULL_9 = np.ascontiguousarray(
    JOINT_TARGET,
    dtype=np.int64
)

assert X_FULL_9.shape == (88186, 10, 14)
assert Y_FULL_9.shape == (88186,)

assert np.isfinite(X_FULL_9).all()
assert X_FULL_9.dtype == np.float32

SCALER_STATE_9 = FINAL_CR_SCALER.state_dict()

assert SCALER_STATE_9["feature_names"] == FEATURES

print("\n✓ Full-TRAIN scaler fitted")
print("Scaled features:", X_FULL_9.shape)

# ==========================================================================================
# 9G — CONSTRUCT TRAIN DATALOADER
# ==========================================================================================

train_dataset_9 = TensorDataset(
    torch.from_numpy(X_FULL_9),
    torch.from_numpy(Y_FULL_9)
)

train_generator_9 = torch.Generator()
train_generator_9.manual_seed(FINAL_SEED_9)

train_loader_9 = DataLoader(
    train_dataset_9,
    batch_size=FINAL_BATCH_SIZE_9,
    shuffle=True,
    generator=train_generator_9,
    num_workers=0,
    pin_memory=True
)

# ==========================================================================================
# 9H — INITIALIZE BRAND-NEW COMPETING-RISK MODEL
# ==========================================================================================

FINAL_CR_CONFIG = {
    "input_dim": 14,
    "hidden_dim": 64,
    "num_layers": 2,
    "dropout": 0.20,
    "dense_dim": 32,
    "num_classes": 76
}

FINAL_CR_MODEL = LSTMCompetingRisk(
    **FINAL_CR_CONFIG
).to(FINAL_DEVICE_9)

FINAL_CR_OPTIMIZER = torch.optim.AdamW(
    FINAL_CR_MODEL.parameters(),
    lr=FINAL_LR_9,
    weight_decay=FINAL_WEIGHT_DECAY_9
)

parameter_count_9 = sum(
    p.numel()
    for p in FINAL_CR_MODEL.parameters()
)

assert parameter_count_9 == 58348

print("\nFinal model architecture:")
print(FINAL_CR_MODEL)

print("\nTrainable parameters:", parameter_count_9)

# ==========================================================================================
# 9I — TRAIN EXACTLY 17 EPOCHS
#
# No CV holdout is used here.
# No early stopping.
# No official validation loader.
# ==========================================================================================

FINAL_CR_HISTORY = []

training_start_9 = time.perf_counter()

for epoch in range(1, FINAL_EPOCHS_9 + 1):

    FINAL_CR_MODEL.train()

    loss_total = 0.0
    seen = 0

    for xb, yb in train_loader_9:

        xb = xb.to(
            FINAL_DEVICE_9,
            non_blocking=True
        )

        yb = yb.to(
            FINAL_DEVICE_9,
            non_blocking=True
        )

        FINAL_CR_OPTIMIZER.zero_grad(
            set_to_none=True
        )

        logits = FINAL_CR_MODEL(xb)

        loss = F.cross_entropy(
            logits,
            yb
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Nonfinite loss at epoch {epoch}."
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            FINAL_CR_MODEL.parameters(),
            max_norm=FINAL_GRAD_CLIP_9,
            error_if_nonfinite=True
        )

        FINAL_CR_OPTIMIZER.step()

        batch_count = len(xb)

        loss_total += (
            float(loss.item()) * batch_count
        )

        seen += batch_count

    epoch_nll = loss_total / seen

    if seen != len(X_RAW):
        raise RuntimeError(
            "Training epoch did not cover all TRAIN windows."
        )

    if not np.isfinite(epoch_nll):
        raise RuntimeError(
            f"Nonfinite epoch NLL: {epoch}"
        )

    elapsed = time.perf_counter() - training_start_9

    FINAL_CR_HISTORY.append({
        "epoch": int(epoch),
        "train_joint_nll": float(epoch_nll),
        "windows_seen": int(seen),
        "elapsed_seconds": float(elapsed)
    })

    print(
        f"Epoch {epoch:02d}/{FINAL_EPOCHS_9} | "
        f"TRAIN joint NLL={epoch_nll:.6f} | "
        f"Windows={seen:,}"
    )

assert len(FINAL_CR_HISTORY) == FINAL_EPOCHS_9

FINAL_CR_HISTORY_DF = pd.DataFrame(
    FINAL_CR_HISTORY
)

print("\n✓ Exactly 17 epochs completed")

# ==========================================================================================
# 9J — NUMERICAL AND PROBABILITY SANITY CHECK
#
# Uses only a small TRAIN sample.
# Not an official performance evaluation.
# ==========================================================================================

FINAL_CR_MODEL.eval()

with torch.inference_mode():

    preview_x_9 = torch.from_numpy(
        X_FULL_9[:256]
    ).to(FINAL_DEVICE_9)

    preview_logits_9 = FINAL_CR_MODEL(
        preview_x_9
    )

    preview_out_9 = competing_risk_outputs(
        preview_logits_9
    )

    joint_p_9 = preview_out_9["joint_probs"]
    horizons_p_9 = preview_out_9["horizon_probs"]

    assert joint_p_9.shape == (256, 76)
    assert horizons_p_9.shape == (256, 3)

    assert torch.isfinite(joint_p_9).all()
    assert torch.isfinite(horizons_p_9).all()

    assert torch.allclose(
        joint_p_9.sum(dim=1),
        torch.ones(256, device=FINAL_DEVICE_9),
        atol=1e-6
    )

    assert torch.all(
        (joint_p_9 >= 0) &
        (joint_p_9 <= 1)
    )

    assert torch.all(
        horizons_p_9[:, 0]
        <= horizons_p_9[:, 1] + 1e-7
    )

    assert torch.all(
        horizons_p_9[:, 1]
        <= horizons_p_9[:, 2] + 1e-7
    )

    assert torch.allclose(
        preview_out_9["no_event_prob"]
        + horizons_p_9[:, 2],
        torch.ones(256, device=FINAL_DEVICE_9),
        atol=1e-6
    )

    assert torch.allclose(
        preview_out_9["family_probs"].sum(dim=1),
        horizons_p_9[:, 2],
        atol=1e-6
    )

    assert torch.allclose(
        preview_out_9["onset_probs"].sum(dim=1),
        horizons_p_9[:, 2],
        atol=1e-6
    )

print("✓ 76-class probabilities valid")
print("✓ H5 <= H10 <= H15")
print("✓ Family and onset marginals valid")
print("✓ No-event complement valid")

# ==========================================================================================
# 9K — CREATE TRAINING RECORD
# ==========================================================================================

training_record_9 = {
    "model_name": "LSTM_V2_COMPETING_RISK",
    "task": "JOINT_ONSET_MINUTE_AND_ATTACK_FAMILY",
    "dataset": "Synthetic_NAF_v2",
    "dataset_version": "2.0.0",
    "dataset_manifest_sha256": canonical_sha256_9(
        manifest_9
    ),
    "train_data_sha256": expected_train_hash_9,
    "training_split": "TRAIN",
    "training_windows": int(len(X_RAW)),
    "training_episodes": int(
        len(np.unique(EPISODE_ID))
    ),
    "input_shape": [10, 14],
    "output_classes": 76,
    "output_definition": {
        "class_0": "NO_ONSET_WITHIN_15_MINUTES",
        "class_1_to_75": (
            "CLASS = 5 * (ONSET_MINUTE - 1) + FAMILY_ID"
        ),
        "onset_minutes": list(range(1, 16)),
        "family_names": list(FAMILIES)
    },
    "model_architecture": FINAL_CR_CONFIG,
    "trainable_parameters": int(parameter_count_9),
    "selected_configuration_id": "C01",
    "selected_configuration": selected_config_9,
    "selection_protocol_hash": selection_9["protocol_hash"],
    "selected_cv_mean_joint_nll": (
        selection_9["selected_mean_joint_nll"]
    ),
    "selected_cv_mean_H5_AP": (
        selection_9["selected_mean_H5_AP"]
    ),
    "selected_cv_mean_H10_AP": (
        selection_9["selected_mean_H10_AP"]
    ),
    "selected_cv_mean_H15_AP": (
        selection_9["selected_mean_H15_AP"]
    ),
    "selected_cv_mean_family_macro_f1": (
        selection_9["selected_mean_family_macro_f1"]
    ),
    "selected_cv_mean_eta_mae": (
        selection_9["selected_mean_eta_mae"]
    ),
    "training_epochs": FINAL_EPOCHS_9,
    "training_seed": FINAL_SEED_9,
    "optimizer": "AdamW",
    "learning_rate": FINAL_LR_9,
    "weight_decay": FINAL_WEIGHT_DECAY_9,
    "batch_size": FINAL_BATCH_SIZE_9,
    "gradient_clip_norm": FINAL_GRAD_CLIP_9,
    "loss_function": "UNWEIGHTED_76_CLASS_CROSS_ENTROPY",
    "preprocessing": (
        "LOG1P_MEDIAN_IQR_MIN_SCALE_0.05_CLIP_8"
    ),
    "feature_names": list(FEATURES),
    "torch_version": torch.__version__,
    "python_version": platform.python_version(),
    "device": str(FINAL_DEVICE_9),
    "training_status": "COMPLETED",
    "official_validation_used": False,
    "official_test_used": False,
    "dataset_modified": False,
    "trained_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

# ==========================================================================================
# 9L — SAVE FINAL ARTIFACTS
#
# Create a staging directory, verify the saved model,
# then rename it to the final release directory.
# ==========================================================================================

STAGING_DIR_9 = FINAL_DIR_9.with_name(
    "final_train_model_STAGING"
)

if STAGING_DIR_9.exists():
    raise RuntimeError(
        "A previous staging directory exists:\n"
        f"{STAGING_DIR_9}\n"
        "Inspect it before attempting another save."
    )

STAGING_DIR_9.mkdir(
    parents=True,
    exist_ok=False
)

MODEL_FILE_9 = (
    STAGING_DIR_9
    / "LSTM_V2_CR_STATE.pt"
)

SCALER_FILE_9 = (
    STAGING_DIR_9
    / "LSTM_V2_CR_FULL_TRAIN_SCALER.json"
)

HISTORY_FILE_9 = (
    STAGING_DIR_9
    / "LSTM_V2_CR_FINAL_TRAIN_HISTORY.csv"
)

RECORD_FILE_9 = (
    STAGING_DIR_9
    / "LSTM_V2_CR_FINAL_TRAIN_RECORD.json"
)

REGISTRY_FILE_9 = (
    STAGING_DIR_9
    / "LSTM_V2_CR_MODEL_REGISTRY.json"
)

# Save CPU state tensors, not an arbitrary Python model object.
cpu_weights_9 = {
    name: tensor.detach().cpu().clone()
    for name, tensor in FINAL_CR_MODEL.state_dict().items()
}

torch.save(
    cpu_weights_9,
    MODEL_FILE_9
)

write_json_9(
    SCALER_FILE_9,
    SCALER_STATE_9
)

FINAL_CR_HISTORY_DF.to_csv(
    HISTORY_FILE_9,
    index=False
)

training_record_9["model_sha256"] = (
    file_sha256_9(MODEL_FILE_9)
)

training_record_9["scaler_sha256"] = (
    file_sha256_9(SCALER_FILE_9)
)

training_record_9["history_sha256"] = (
    file_sha256_9(HISTORY_FILE_9)
)

write_json_9(
    RECORD_FILE_9,
    training_record_9
)

registry_9 = {
    "dataset": "Synthetic_NAF_v2",
    "dataset_version": "2.0.0",
    "model_name": "LSTM_V2_COMPETING_RISK",
    "training_status": "FINAL_FULL_TRAIN_COMPLETED",
    "validation_status": "NOT_STARTED",
    "test_status": "EMBARGOED",
    "selected_configuration": "C01",
    "training_epochs": 17,
    "model_file": MODEL_FILE_9.name,
    "model_sha256": training_record_9["model_sha256"],
    "scaler_file": SCALER_FILE_9.name,
    "scaler_sha256": training_record_9["scaler_sha256"],
    "history_file": HISTORY_FILE_9.name,
    "history_sha256": training_record_9["history_sha256"],
    "training_record_file": RECORD_FILE_9.name,
    "training_record_sha256": file_sha256_9(
        RECORD_FILE_9
    ),
    "feature_names": list(FEATURES),
    "output_classes": 76,
    "forecast_horizons": [5, 10, 15]
}

write_json_9(
    REGISTRY_FILE_9,
    registry_9
)

# ==========================================================================================
# 9M — VERIFY SAVED WEIGHTS BY RELOADING
# ==========================================================================================

saved_state_9 = torch.load(
    MODEL_FILE_9,
    map_location="cpu",
    weights_only=True
)

RELOADED_CR_MODEL_9 = LSTMCompetingRisk(
    **FINAL_CR_CONFIG
).cpu()

RELOADED_CR_MODEL_9.load_state_dict(
    saved_state_9,
    strict=True
)

RELOADED_CR_MODEL_9.eval()
FINAL_CR_MODEL.eval()

with torch.inference_mode():

    test_input_cpu_9 = torch.from_numpy(
        X_FULL_9[:32]
    )

    original_logits_9 = FINAL_CR_MODEL(
        test_input_cpu_9.to(FINAL_DEVICE_9)
    ).cpu()

    reloaded_logits_9 = RELOADED_CR_MODEL_9(
        test_input_cpu_9
    )

    # Compare probabilities across CPU/GPU.
    original_probs_9 = torch.softmax(
        original_logits_9,
        dim=1
    )

    reloaded_probs_9 = torch.softmax(
        reloaded_logits_9,
        dim=1
    )

    if not torch.allclose(
        original_probs_9,
        reloaded_probs_9,
        rtol=1e-5,
        atol=1e-5
    ):
        raise RuntimeError(
            "Saved model reload prediction mismatch."
        )

assert (
    file_sha256_9(MODEL_FILE_9)
    == registry_9["model_sha256"]
)

assert (
    file_sha256_9(SCALER_FILE_9)
    == registry_9["scaler_sha256"]
)

assert (
    file_sha256_9(RECORD_FILE_9)
    == registry_9["training_record_sha256"]
)

print("\n✓ Saved model reloaded successfully")
print("✓ Reloaded probabilities match original model")
print("✓ Model/scaler/record hashes verified")

# ==========================================================================================
# 9N — FINALIZE RELEASE DIRECTORY
# ==========================================================================================

# Ensure another notebook/process hasn't created the final folder.
if FINAL_DIR_9.exists():
    raise RuntimeError(
        "Final directory appeared during training. "
        "Staging artifacts were preserved; refusing overwrite."
    )

STAGING_DIR_9.rename(FINAL_DIR_9)

# Verify the final files exist after directory rename.
for filename in [
    "LSTM_V2_CR_STATE.pt",
    "LSTM_V2_CR_FULL_TRAIN_SCALER.json",
    "LSTM_V2_CR_FINAL_TRAIN_HISTORY.csv",
    "LSTM_V2_CR_FINAL_TRAIN_RECORD.json",
    "LSTM_V2_CR_MODEL_REGISTRY.json"
]:
    assert (FINAL_DIR_9 / filename).is_file()

# ==========================================================================================
# 9O — FINAL TRAINING REPORT
# ==========================================================================================

print("\n" + "=" * 100)
print("COMPETING-RISK LSTM — FINAL FULL-TRAIN FITTING COMPLETE")
print("=" * 100)

print("\nSelected configuration: C01")
print("Training windows:", len(X_RAW))
print("Training episodes:", len(np.unique(EPISODE_ID)))
print("Output classes:", 76)
print("Trainable parameters:", parameter_count_9)
print("Epochs completed:", len(FINAL_CR_HISTORY_DF))

print(
    "\nInitial epoch joint NLL:",
    round(
        float(
            FINAL_CR_HISTORY_DF.iloc[0]["train_joint_nll"]
        ),
        6
    )
)

print(
    "Final epoch joint NLL:",
    round(
        float(
            FINAL_CR_HISTORY_DF.iloc[-1]["train_joint_nll"]
        ),
        6
    )
)

print("\nModel saved:")
print(
    FINAL_DIR_9
    / "LSTM_V2_CR_STATE.pt"
)

print("\nScaler saved:")
print(
    FINAL_DIR_9
    / "LSTM_V2_CR_FULL_TRAIN_SCALER.json"
)

print("\nFinal model SHA256:")
print(training_record_9["model_sha256"])

print("\nFinal competing-risk LSTM: TRAINED AND SAVED")
print("Additional hyperparameter selection: NONE")
print("Official VALIDATION: NOT ACCESSED")
print("Official TEST: NOT ACCESSED")
print("Frozen dataset: UNCHANGED")

print("\nNEXT: Official competing-risk VALIDATION")
print("=" * 100)


In [ ]:

# ==========================================================================================
# LSTM V2 COMPETING-RISK — CELL 10
# OFFICIAL VALIDATION OF THE FROZEN FINAL C01 MODEL
#
# READ-ONLY EVALUATION
#
# Evaluates:
#   1. Joint 76-class NLL
#   2. H5/H10/H15 AP, ROC-AUC, Brier, 10-bin ECE
#   3. Family classification conditional on true future onset
#   4. Exact-minute and expected-ETA performance
#   5. End-to-end gated family predictions
#   6. Source-specific metrics
#   7. Benign false-alert statistics
#   8. Episode-cluster bootstrap confidence intervals
#   9. Historical benchmark comparison
#
# NO model training, tuning, threshold optimization or recalibration.
# NO official TEST access.
# NO modification of frozen data or saved model.
# ==========================================================================================

import json
import hashlib
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    accuracy_score,
    f1_score,
    confusion_matrix,
    precision_recall_fscore_support
)

# ==========================================================================================
# 10A — PATHS AND LOCKED SETTINGS
# ==========================================================================================

ROOT_10 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATA_DIR_10 = (
    ROOT_10 / "Synthetic_NAF_v2/release_candidate_corrected"
)

FREEZE_DIR_10 = (
    ROOT_10 / "Synthetic_NAF_v2/FINAL_RELEASE_FREEZE"
)

MODEL_DIR_10 = (
    ROOT_10
    / "Synthetic_NAF_v2/models/LSTM_V2_CompetingRisk/final_train_model"
)

CV_DIR_10 = (
    ROOT_10
    / "Synthetic_NAF_v2/models/LSTM_V2_CompetingRisk/train_cv_selection_GPU"
)

REPORT_DIR_10 = (
    ROOT_10
    / "Synthetic_NAF_v2/models/LSTM_V2_CompetingRisk/official_validation"
)

VAL_FILE_10 = DATA_DIR_10 / "windows_W10_VALIDATION.npz"

META_FILE_10 = (
    DATA_DIR_10 / "episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv"
)

MANIFEST_FILE_10 = (
    FREEZE_DIR_10 / "Synthetic_NAF_v2_RELEASE_MANIFEST.json"
)

LOCK_FILE_10 = (
    FREEZE_DIR_10 / "Synthetic_NAF_v2_FREEZE_LOCK.json"
)

MODEL_FILE_10 = MODEL_DIR_10 / "LSTM_V2_CR_STATE.pt"

SCALER_FILE_10 = (
    MODEL_DIR_10 / "LSTM_V2_CR_FULL_TRAIN_SCALER.json"
)

RECORD_FILE_10 = (
    MODEL_DIR_10 / "LSTM_V2_CR_FINAL_TRAIN_RECORD.json"
)

REGISTRY_FILE_10 = (
    MODEL_DIR_10 / "LSTM_V2_CR_MODEL_REGISTRY.json"
)

SELECTION_FILE_10 = (
    CV_DIR_10 / "SELECTED_CR_CONFIGURATION.json"
)

PROTOCOL_FILE_10 = (
    CV_DIR_10 / "SEARCH_PROTOCOL.json"
)

EXPECTED_MODEL_SHA256_10 = (
    "062f8dda7b0045c605bb4b2e52de927"
    "e576661c88e4c7fb9009fb367fb9717fd"
)

HORIZONS_10 = [5, 10, 15]

FIXED_THRESHOLDS_10 = [0.10, 0.20, 0.50]

BOOTSTRAP_REPLICATES_10 = 300
BOOTSTRAP_SEED_10 = 20261024

INFERENCE_BATCH_10 = 1024

# Protect results from accidental repeated official evaluation.
if REPORT_DIR_10.exists():
    raise RuntimeError(
        "Official competing-risk validation directory already exists:\n"
        f"{REPORT_DIR_10}\n"
        "Refusing to overwrite previous results."
    )

# ==========================================================================================
# 10B — HELPERS
# ==========================================================================================

def load_json_10(path):
    if not path.is_file():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding="utf-8"))


def sha256_file_10(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(
            lambda: stream.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()


def canonical_hash_10(obj):
    data = json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    ).encode("utf-8")
    return hashlib.sha256(data).hexdigest()


def write_json_10(path, data):
    path.write_text(
        json.dumps(
            data, indent=2, sort_keys=True, allow_nan=False
        ),
        encoding="utf-8"
    )


def to_bool_10(values):
    text = (
        pd.Series(values)
        .astype("string")
        .str.strip()
        .str.lower()
    )
    yes = {"true", "1", "yes"}
    no = {"false", "0", "no"}
    if not text.isin(yes | no).all():
        raise ValueError(
            "Unrecognized Hard_Benign metadata values."
        )
    return text.isin(yes).to_numpy(dtype=bool)


def ece_10(y, p, n_bins=10):
    y = np.asarray(y, dtype=np.float64)
    p = np.asarray(p, dtype=np.float64)

    ids = np.minimum(
        (p * n_bins).astype(int),
        n_bins - 1
    )

    total = len(y)
    error = 0.0
    calibration_rows = []

    for b in range(n_bins):
        mask = ids == b
        count = int(mask.sum())

        if count:
            conf = float(p[mask].mean())
            observed = float(y[mask].mean())
            error += (
                count / total
            ) * abs(conf - observed)
        else:
            conf = None
            observed = None

        calibration_rows.append({
            "bin": b,
            "lower": b / n_bins,
            "upper": (b + 1) / n_bins,
            "count": count,
            "mean_prediction": conf,
            "observed_rate": observed
        })

    return float(error), calibration_rows


def binary_metrics_10(y, p):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)

    if len(np.unique(y)) != 2:
        return None

    ece, bins = ece_10(y, p)

    return {
        "windows": int(len(y)),
        "prevalence": float(y.mean()),
        "AP": float(average_precision_score(y, p)),
        "ROC_AUC": float(roc_auc_score(y, p)),
        "Brier": float(brier_score_loss(y, p)),
        "ECE_10_bins": float(ece)
    }, bins


# ==========================================================================================
# 10C — CHECKPOINT AND DATASET INTEGRITY
# ==========================================================================================

if "LSTMCompetingRisk" not in globals():
    raise RuntimeError(
        "Run Cell 5 to define LSTMCompetingRisk before validation."
    )

manifest_10 = load_json_10(MANIFEST_FILE_10)
lock_10 = load_json_10(LOCK_FILE_10)

record_10 = load_json_10(RECORD_FILE_10)
registry_10 = load_json_10(REGISTRY_FILE_10)
scaler_state_10 = load_json_10(SCALER_FILE_10)

selection_10 = load_json_10(SELECTION_FILE_10)
protocol_10 = load_json_10(PROTOCOL_FILE_10)

assert manifest_10["release_version"] == "2.0.0"

assert (
    canonical_hash_10(manifest_10)
    == lock_10["manifest_sha256_canonical"]
)

assert (
    canonical_hash_10(protocol_10)
    == selection_10["protocol_hash"]
)

assert selection_10["selected_config_id"] == "C01"
assert selection_10["selected_final_epochs"] == 17

assert record_10["training_epochs"] == 17
assert record_10["training_windows"] == 88186
assert record_10["training_episodes"] == 1400

assert record_10["official_validation_used"] is False
assert record_10["official_test_used"] is False

assert registry_10["validation_status"] == "NOT_STARTED"
assert registry_10["test_status"] == "EMBARGOED"

assert registry_10["output_classes"] == 76
assert registry_10["selected_configuration"] == "C01"

assert record_10["selection_protocol_hash"] == (
    selection_10["protocol_hash"]
)

model_hash_10 = sha256_file_10(MODEL_FILE_10)

if model_hash_10 != EXPECTED_MODEL_SHA256_10:
    raise RuntimeError(
        "FINAL COMPETING-RISK MODEL HASH MISMATCH."
    )

assert model_hash_10 == registry_10["model_sha256"]
assert model_hash_10 == record_10["model_sha256"]

assert sha256_file_10(SCALER_FILE_10) == (
    registry_10["scaler_sha256"]
)

assert sha256_file_10(RECORD_FILE_10) == (
    registry_10["training_record_sha256"]
)

assert sha256_file_10(
    DATA_DIR_10 / "windows_W10_TRAIN.npz"
) == record_10["train_data_sha256"]

expected_val_hash_10 = (
    manifest_10["data_integrity"]["file_sha256"]
    ["windows_W10_VALIDATION.npz"]
)

if sha256_file_10(VAL_FILE_10) != expected_val_hash_10:
    raise RuntimeError(
        "Frozen official VALIDATION hash mismatch."
    )

expected_meta_hash_10 = (
    manifest_10["data_integrity"]["file_sha256"]
    [META_FILE_10.name]
)

if sha256_file_10(META_FILE_10) != expected_meta_hash_10:
    raise RuntimeError("Metadata hash mismatch.")

FEATURE_NAMES_10 = record_10["feature_names"]

FAMILY_NAMES_10 = record_10[
    "output_definition"
]["family_names"]

assert len(FEATURE_NAMES_10) == 14
assert len(FAMILY_NAMES_10) == 5

assert FEATURE_NAMES_10 == registry_10["feature_names"]
assert FEATURE_NAMES_10 == scaler_state_10["feature_names"]

print("=" * 100)
print("COMPETING-RISK LSTM — OFFICIAL VALIDATION")
print("=" * 100)

print("Release:", manifest_10["release_version"])
print("Selected model:", "C01")
print("Model hash: VERIFIED")
print("Scaler hash: VERIFIED")
print("Training record: VERIFIED")
print("Frozen validation data: VERIFIED")
print("Training: DISABLED")
print("Official TEST: NOT ACCESSED")

# ==========================================================================================
# 10D — LOAD FROZEN OFFICIAL VALIDATION
# ==========================================================================================

with np.load(VAL_FILE_10, allow_pickle=False) as pack:

    X_VAL_RAW_10 = pack["X"].astype(np.float32)

    Y_VAL_10 = pack["y_horizons"].astype(np.int8)

    ONSET_VAL_10 = pack["onset_class"].astype(np.int16)

    FAMILY_VAL_10 = pack["family_class"].astype(np.int16)

    EPISODE_VAL_10 = (
        pd.Series(pack["episode_id"])
        .astype(str)
        .to_numpy()
    )

    MINUTE_VAL_10 = pack["minute"].astype(np.int16)

assert X_VAL_RAW_10.shape == (18839, 10, 14)
assert Y_VAL_10.shape == (18839, 3)

assert len(np.unique(EPISODE_VAL_10)) == 300

assert np.isfinite(X_VAL_RAW_10).all()
assert np.all(X_VAL_RAW_10 >= 0)

assert np.all((ONSET_VAL_10 >= 0) & (ONSET_VAL_10 <= 15))
assert np.all((FAMILY_VAL_10 >= 0) & (FAMILY_VAL_10 <= 5))

assert np.array_equal(
    ONSET_VAL_10 == 0,
    FAMILY_VAL_10 == 0
)

for col, horizon in enumerate(HORIZONS_10):
    reconstructed = (
        (ONSET_VAL_10 >= 1)
        & (ONSET_VAL_10 <= horizon)
    ).astype(np.int8)

    assert np.array_equal(
        reconstructed,
        Y_VAL_10[:, col]
    )

# Same exact joint-target encoding used during TRAIN.
JOINT_VAL_10 = np.where(
    ONSET_VAL_10 == 0,
    0,
    (ONSET_VAL_10 - 1) * 5 + FAMILY_VAL_10
).astype(np.int64)

assert np.all((JOINT_VAL_10 >= 0) & (JOINT_VAL_10 <= 75))

# Metadata is used only for auditing/subgroup evaluation.
# It must never enter the model input.
metadata_10 = pd.read_csv(META_FILE_10)

metadata_10["Episode_ID"] = (
    metadata_10["Episode_ID"].astype(str)
)

assert metadata_10["Episode_ID"].is_unique

val_metadata_10 = (
    metadata_10
    .set_index("Episode_ID")
    .loc[EPISODE_VAL_10]
)

assert set(
    val_metadata_10["Split"].astype(str)
) == {"VALIDATION"}

SOURCE_VAL_10 = (
    val_metadata_10["Source"].astype(str).to_numpy()
)

FAMILY_NAME_VAL_10 = (
    val_metadata_10["Family"].astype(str).to_numpy()
)

HARD_BENIGN_VAL_10 = to_bool_10(
    val_metadata_10["Hard_Benign"]
)

IS_WHOLE_BENIGN_10 = (
    FAMILY_NAME_VAL_10 == "BENIGN"
)

assert set(np.unique(SOURCE_VAL_10)) == {
    "CIC_IDS2017", "UNSW_NB15"
}

assert not np.any(
    HARD_BENIGN_VAL_10 & ~IS_WHOLE_BENIGN_10
)

# Wholly benign episodes must have no within-horizon onset.
assert np.all(
    JOINT_VAL_10[IS_WHOLE_BENIGN_10] == 0
)

print("\nOFFICIAL VALIDATION DATA")
print("Input shape:", X_VAL_RAW_10.shape)
print("Episodes:", len(np.unique(EPISODE_VAL_10)))
print("Sources:", sorted(set(SOURCE_VAL_10)))
print("Future-event windows:", int((JOINT_VAL_10 > 0).sum()))
print("No-onset windows:", int((JOINT_VAL_10 == 0).sum()))

for col, horizon in enumerate(HORIZONS_10):
    print(
        f"H{horizon} prevalence:",
        f"{Y_VAL_10[:, col].mean():.6f}"
    )

# ==========================================================================================
# 10E — APPLY THE EXACT SAVED FULL-TRAIN SCALER
#
# No fitting on VALIDATION.
# ==========================================================================================

median_10 = np.asarray(
    scaler_state_10["median"],
    dtype=np.float32
)

scale_10 = np.asarray(
    scaler_state_10["scale"],
    dtype=np.float32
)

clip_10 = float(
    scaler_state_10["clip_value"]
)

assert median_10.shape == (14,)
assert scale_10.shape == (14,)

assert np.isfinite(median_10).all()
assert np.isfinite(scale_10).all()
assert np.all(scale_10 > 0)
assert clip_10 == 8.0

X_VAL_LOG_10 = np.log1p(
    X_VAL_RAW_10
)

X_VAL_SCALED_10 = (
    X_VAL_LOG_10
    - median_10[None, None, :]
) / scale_10[None, None, :]

X_VAL_SCALED_10 = np.ascontiguousarray(
    np.clip(X_VAL_SCALED_10, -clip_10, clip_10),
    dtype=np.float32
)

assert X_VAL_SCALED_10.shape == (18839, 10, 14)
assert np.isfinite(X_VAL_SCALED_10).all()

print("\n✓ Saved TRAIN scaler applied")
print("✓ No scaler refitting")

# ==========================================================================================
# 10F — RELOAD FROZEN COMPETING-RISK MODEL
# ==========================================================================================

architecture_10 = record_10["model_architecture"]

assert architecture_10 == {
    "input_dim": 14,
    "hidden_dim": 64,
    "num_layers": 2,
    "dropout": 0.2,
    "dense_dim": 32,
    "num_classes": 76
}

DEVICE_10 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

EVAL_MODEL_10 = LSTMCompetingRisk(
    **architecture_10
).to(DEVICE_10)

saved_weights_10 = torch.load(
    MODEL_FILE_10,
    map_location="cpu",
    weights_only=True
)

EVAL_MODEL_10.load_state_dict(
    saved_weights_10,
    strict=True
)

EVAL_MODEL_10.eval()
EVAL_MODEL_10.requires_grad_(False)

assert not EVAL_MODEL_10.training
assert all(
    not p.requires_grad
    for p in EVAL_MODEL_10.parameters()
)

loader_10 = DataLoader(
    TensorDataset(
        torch.from_numpy(X_VAL_SCALED_10)
    ),
    batch_size=INFERENCE_BATCH_10,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE_10.type == "cuda")
)

# ==========================================================================================
# 10G — GENERATE JOINT 76-CLASS PROBABILITIES
# ==========================================================================================

joint_parts_10 = []

with torch.inference_mode():

    for (xb,) in loader_10:

        xb = xb.to(
            DEVICE_10,
            non_blocking=True
        )

        logits = EVAL_MODEL_10(xb)

        assert logits.shape[1] == 76
        assert torch.isfinite(logits).all()

        probs = torch.softmax(
            logits,
            dim=1
        )

        joint_parts_10.append(
            probs.cpu().numpy().astype(np.float64)
        )

JOINT_P_10 = np.concatenate(
    joint_parts_10,
    axis=0
)

assert JOINT_P_10.shape == (18839, 76)

assert np.isfinite(JOINT_P_10).all()
assert np.all(JOINT_P_10 >= 0)
assert np.all(JOINT_P_10 <= 1)

assert np.allclose(
    JOINT_P_10.sum(axis=1),
    1.0,
    atol=1e-6
)

# Shape: 18,839 windows x 15 onset minutes x 5 families.
EVENT_MATRIX_10 = JOINT_P_10[:, 1:].reshape(
    -1, 15, 5
)

NO_EVENT_P_10 = JOINT_P_10[:, 0]

ONSET_P_10 = EVENT_MATRIX_10.sum(axis=2)
FAMILY_P_10 = EVENT_MATRIX_10.sum(axis=1)

# P(onset within H5/H10/H15).
cumulative_10 = ONSET_P_10.cumsum(axis=1)

HORIZON_P_10 = cumulative_10[:, [4, 9, 14]]

EVENT_15_P_10 = HORIZON_P_10[:, 2]

assert np.allclose(
    NO_EVENT_P_10 + EVENT_15_P_10,
    1.0,
    atol=1e-6
)

assert np.allclose(
    FAMILY_P_10.sum(axis=1),
    EVENT_15_P_10,
    atol=1e-6
)

assert np.allclose(
    ONSET_P_10.sum(axis=1),
    EVENT_15_P_10,
    atol=1e-6
)

violations_5_10 = int(np.sum(
    HORIZON_P_10[:, 0]
    > HORIZON_P_10[:, 1] + 1e-7
))

violations_10_15 = int(np.sum(
    HORIZON_P_10[:, 1]
    > HORIZON_P_10[:, 2] + 1e-7
))

assert violations_5_10 == 0
assert violations_10_15 == 0

# Conditional family/onset distributions.
# Meaningful only when an event is conditioned on.
conditional_denominator_10 = np.maximum(
    EVENT_15_P_10,
    1e-12
)

FAMILY_COND_P_10 = (
    FAMILY_P_10 /
    conditional_denominator_10[:, None]
)

ONSET_COND_P_10 = (
    ONSET_P_10 /
    conditional_denominator_10[:, None]
)

assert np.allclose(
    FAMILY_COND_P_10.sum(axis=1),
    1.0,
    atol=1e-5
)

assert np.allclose(
    ONSET_COND_P_10.sum(axis=1),
    1.0,
    atol=1e-5
)

print("\n✓ Frozen model inference complete")
print("✓ Joint probabilities sum to 1")
print("✓ Horizon ordering verified")
print("✓ Family/onset marginals verified")

# ==========================================================================================
# 10H — JOINT 76-CLASS NEGATIVE LOG-LIKELIHOOD
# ==========================================================================================

row_indices_10 = np.arange(
    len(JOINT_VAL_10)
)

true_joint_prob_10 = JOINT_P_10[
    row_indices_10,
    JOINT_VAL_10
]

# Float32 softmax can underflow on extremely unlikely classes.
# Clip only inside log calculation.
JOINT_NLL_10 = float(
    -np.log(
        np.clip(
            true_joint_prob_10,
            1e-30,
            1.0
        )
    ).mean()
)

JOINT_TOP1_10 = float(
    accuracy_score(
        JOINT_VAL_10,
        JOINT_P_10.argmax(axis=1)
    )
)

print("\nJOINT 76-CLASS PERFORMANCE")
print("Joint NLL:", round(JOINT_NLL_10, 6))
print("Joint top-1 accuracy:", round(JOINT_TOP1_10, 6))

# ==========================================================================================
# 10I — H5/H10/H15 ATTACK FORECASTING
# ==========================================================================================

overall_rows_10 = []
calibration_rows_10 = []

for col, horizon in enumerate(HORIZONS_10):

    metrics, bins = binary_metrics_10(
        Y_VAL_10[:, col],
        HORIZON_P_10[:, col]
    )

    if metrics is None:
        raise RuntimeError(
            f"H{horizon}: only one ground-truth class."
        )

    overall_rows_10.append({
        "Horizon": f"H{horizon}",
        **metrics
    })

    for item in bins:
        calibration_rows_10.append({
            "Horizon": f"H{horizon}",
            **item
        })

OVERALL_DF_10 = pd.DataFrame(
    overall_rows_10
)

CALIBRATION_DF_10 = pd.DataFrame(
    calibration_rows_10
)

# ==========================================================================================
# 10J — FAMILY CLASSIFICATION
#
# Oracle-conditional:
# We evaluate family on windows where the ground truth
# has an onset within 15 minutes.
#
# This is NOT end-to-end alert performance.
# ==========================================================================================

TRUE_EVENT_MASK_10 = (
    JOINT_VAL_10 > 0
)

true_family_event_10 = FAMILY_VAL_10[
    TRUE_EVENT_MASK_10
]

pred_family_all_10 = (
    FAMILY_COND_P_10.argmax(axis=1) + 1
)

pred_family_event_10 = pred_family_all_10[
    TRUE_EVENT_MASK_10
]

FAMILY_ACCURACY_10 = float(
    accuracy_score(
        true_family_event_10,
        pred_family_event_10
    )
)

FAMILY_MACRO_F1_10 = float(
    f1_score(
        true_family_event_10,
        pred_family_event_10,
        labels=[1, 2, 3, 4, 5],
        average="macro",
        zero_division=0
    )
)

precision_f_10, recall_f_10, f1_f_10, support_f_10 = (
    precision_recall_fscore_support(
        true_family_event_10,
        pred_family_event_10,
        labels=[1, 2, 3, 4, 5],
        zero_division=0
    )
)

FAMILY_PER_CLASS_DF_10 = pd.DataFrame({
    "Family_ID": [1, 2, 3, 4, 5],
    "Family": FAMILY_NAMES_10,
    "Support": support_f_10,
    "Precision": precision_f_10,
    "Recall": recall_f_10,
    "F1": f1_f_10
})

FAMILY_CM_10 = confusion_matrix(
    true_family_event_10,
    pred_family_event_10,
    labels=[1, 2, 3, 4, 5]
)

FAMILY_CM_DF_10 = pd.DataFrame(
    FAMILY_CM_10,
    index=[
        f"True_{name}"
        for name in FAMILY_NAMES_10
    ],
    columns=[
        f"Pred_{name}"
        for name in FAMILY_NAMES_10
    ]
)

# ==========================================================================================
# 10K — ONSET-TIME PERFORMANCE
#
# Only windows with true onset in 1..15.
# The expected ETA is conditional on an event.
# ==========================================================================================

minutes_10 = np.arange(
    1, 16,
    dtype=np.float64
)

EXPECTED_ETA_10 = (
    ONSET_COND_P_10 @ minutes_10
)

PREDICTED_MINUTE_10 = (
    ONSET_COND_P_10.argmax(axis=1) + 1
)

true_minutes_event_10 = (
    ONSET_VAL_10[TRUE_EVENT_MASK_10]
)

expected_eta_event_10 = (
    EXPECTED_ETA_10[TRUE_EVENT_MASK_10]
)

predicted_minute_event_10 = (
    PREDICTED_MINUTE_10[TRUE_EVENT_MASK_10]
)

ETA_MAE_10 = float(
    np.mean(
        np.abs(
            expected_eta_event_10
            - true_minutes_event_10
        )
    )
)

ETA_RMSE_10 = float(
    np.sqrt(
        np.mean(
            (
                expected_eta_event_10
                - true_minutes_event_10
            ) ** 2
        )
    )
)

EXACT_MINUTE_ACCURACY_10 = float(
    np.mean(
        predicted_minute_event_10
        == true_minutes_event_10
    )
)

WITHIN_ONE_MINUTE_10 = float(
    np.mean(
        np.abs(
            predicted_minute_event_10
            - true_minutes_event_10
        ) <= 1
    )
)

WITHIN_TWO_MINUTES_10 = float(
    np.mean(
        np.abs(
            predicted_minute_event_10
            - true_minutes_event_10
        ) <= 2
    )
)

# Diagnostic error breakdown by actual onset minute.
ETA_BY_MINUTE_ROWS_10 = []

for minute in range(1, 16):

    mask = (
        TRUE_EVENT_MASK_10
        & (ONSET_VAL_10 == minute)
    )

    ETA_BY_MINUTE_ROWS_10.append({
        "True_Onset_Minute": minute,
        "Windows": int(mask.sum()),
        "Expected_ETA_MAE": (
            float(
                np.mean(
                    np.abs(
                        EXPECTED_ETA_10[mask] - minute
                    )
                )
            )
            if mask.any() else None
        ),
        "Exact_Minute_Accuracy": (
            float(
                np.mean(
                    PREDICTED_MINUTE_10[mask] == minute
                )
            )
            if mask.any() else None
        )
    })

ETA_BY_MINUTE_DF_10 = pd.DataFrame(
    ETA_BY_MINUTE_ROWS_10
)

# ==========================================================================================
# 10L — FIXED-THRESHOLD END-TO-END EVALUATION
#
# A family prediction counts as correct only if:
#   1. The model actually issues an attack warning.
#   2. The predicted family matches the true family.
#
# Thresholds are diagnostic, not tuned on VALIDATION.
# ==========================================================================================

threshold_rows_10 = []
end_to_end_rows_10 = []

for col, horizon in enumerate(HORIZONS_10):

    y = Y_VAL_10[:, col]
    p = HORIZON_P_10[:, col]

    for threshold in FIXED_THRESHOLDS_10:

        alerted = p >= threshold

        tn, fp, fn, tp = confusion_matrix(
            y,
            alerted.astype(int),
            labels=[0, 1]
        ).ravel()

        precision = (
            float(tp / (tp + fp))
            if tp + fp else 0.0
        )

        recall = (
            float(tp / (tp + fn))
            if tp + fn else 0.0
        )

        fpr = (
            float(fp / (fp + tn))
            if fp + tn else 0.0
        )

        threshold_rows_10.append({
            "Horizon": f"H{horizon}",
            "Threshold": threshold,
            "TP": int(tp),
            "FP": int(fp),
            "TN": int(tn),
            "FN": int(fn),
            "Precision": precision,
            "Recall": recall,
            "False_Positive_Rate": fpr
        })

        # For horizon h, the true event must begin by h.
        true_positive_event = (
            Y_VAL_10[:, col] == 1
        )

        correct_family_alerts = int(
            np.sum(
                alerted
                & true_positive_event
                & (
                    pred_family_all_10
                    == FAMILY_VAL_10
                )
            )
        )

        total_true_events = int(
            true_positive_event.sum()
        )

        family_correct_event_recall = (
            correct_family_alerts / total_true_events
            if total_true_events else 0.0
        )

        family_accuracy_among_detected = (
            correct_family_alerts / tp
            if tp else 0.0
        )

        end_to_end_rows_10.append({
            "Horizon": f"H{horizon}",
            "Threshold": threshold,
            "True_Event_Windows": total_true_events,
            "Detected_Event_Windows": int(tp),
            "Detected_With_Correct_Family": (
                correct_family_alerts
            ),
            "Correct_Family_Event_Recall": (
                float(family_correct_event_recall)
            ),
            "Family_Accuracy_Among_Detected": (
                float(family_accuracy_among_detected)
            )
        })

THRESHOLD_DF_10 = pd.DataFrame(
    threshold_rows_10
)

END_TO_END_DF_10 = pd.DataFrame(
    end_to_end_rows_10
)

# ==========================================================================================
# 10M — BENIGN FALSE ALERTS
#
# Counts wholly benign EPISODES separately.
# These do not include negative windows from attack episodes.
# ==========================================================================================

benign_rows_10 = []

for col, horizon in enumerate(HORIZONS_10):

    for threshold in FIXED_THRESHOLDS_10:

        alerted = (
            HORIZON_P_10[:, col] >= threshold
        )

        categories = {
            "HARD_BENIGN": (
                IS_WHOLE_BENIGN_10
                & HARD_BENIGN_VAL_10
            ),
            "ORDINARY_BENIGN": (
                IS_WHOLE_BENIGN_10
                & ~HARD_BENIGN_VAL_10
            ),
            "ALL_BENIGN": IS_WHOLE_BENIGN_10
        }

        for label, mask in categories.items():

            episode_set = np.unique(
                EPISODE_VAL_10[mask]
            )

            alerted_set = np.unique(
                EPISODE_VAL_10[
                    mask & alerted
                ]
            )

            benign_rows_10.append({
                "Horizon": f"H{horizon}",
                "Threshold": threshold,
                "Category": label,
                "Episodes": int(len(episode_set)),
                "Episodes_With_Alert": int(
                    len(alerted_set)
                ),
                "Alert_Windows": int(
                    np.sum(mask & alerted)
                ),
                "Episode_Alert_Fraction": (
                    float(
                        len(alerted_set)
                        / len(episode_set)
                    )
                    if len(episode_set)
                    else 0.0
                )
            })

BENIGN_DF_10 = pd.DataFrame(
    benign_rows_10
)

# ==========================================================================================
# 10N — SOURCE-SPECIFIC PERFORMANCE
# ==========================================================================================

source_rows_10 = []

for source in sorted(set(SOURCE_VAL_10)):

    mask = (
        SOURCE_VAL_10 == source
    )

    source_event = (
        mask & TRUE_EVENT_MASK_10
    )

    for col, horizon in enumerate(HORIZONS_10):

        metrics_and_bins = binary_metrics_10(
            Y_VAL_10[mask, col],
            HORIZON_P_10[mask, col]
        )

        if metrics_and_bins is None:
            raise RuntimeError(
                f"Source {source}, H{horizon}: one class."
            )

        metrics, _ = metrics_and_bins

        source_rows_10.append({
            "Source": source,
            "Horizon": f"H{horizon}",
            "Episodes": int(
                len(np.unique(EPISODE_VAL_10[mask]))
            ),
            **metrics
        })

    print(
        f"\n{source}: "
        f"event-positive windows = {source_event.sum()}"
    )

SOURCE_DF_10 = pd.DataFrame(
    source_rows_10
)

SOURCE_FAMILY_ROWS_10 = []

for source in sorted(set(SOURCE_VAL_10)):

    mask = (
        (SOURCE_VAL_10 == source)
        & TRUE_EVENT_MASK_10
    )

    true_f = FAMILY_VAL_10[mask]
    pred_f = pred_family_all_10[mask]

    SOURCE_FAMILY_ROWS_10.append({
        "Source": source,
        "Event_Windows": int(mask.sum()),
        "Family_Accuracy": float(
            accuracy_score(true_f, pred_f)
        ),
        "Family_Macro_F1": float(
            f1_score(
                true_f,
                pred_f,
                labels=[1, 2, 3, 4, 5],
                average="macro",
                zero_division=0
            )
        ),
        "ETA_MAE": float(
            np.mean(
                np.abs(
                    EXPECTED_ETA_10[mask]
                    - ONSET_VAL_10[mask]
                )
            )
        )
    })

SOURCE_FAMILY_DF_10 = pd.DataFrame(
    SOURCE_FAMILY_ROWS_10
)

# Note: source-level family macro-F1 may include families absent
# from that source. Inspect support before interpreting these figures.

# ==========================================================================================
# 10O — EPISODE-CLUSTER BOOTSTRAP CONFIDENCE INTERVALS
#
# Same sampled episodes across all horizons and tasks.
# Does not fully account for shared calibration-group dependence.
# ==========================================================================================

rng_10 = np.random.default_rng(
    BOOTSTRAP_SEED_10
)

unique_episodes_10 = np.unique(
    EPISODE_VAL_10
)

episode_indices_10 = [
    np.flatnonzero(
        EPISODE_VAL_10 == eid
    )
    for eid in unique_episodes_10
]

bootstrap_10 = {
    "Joint_NLL": [],
    "Family_Macro_F1": [],
    "ETA_MAE": [],
    "H5_AP": [],
    "H10_AP": [],
    "H15_AP": []
}

for rep in range(
    BOOTSTRAP_REPLICATES_10
):

    sampled_episodes = rng_10.integers(
        0,
        len(unique_episodes_10),
        size=len(unique_episodes_10)
    )

    idx = np.concatenate([
        episode_indices_10[position]
        for position in sampled_episodes
    ])

    if not all(
        len(np.unique(Y_VAL_10[idx, col])) == 2
        for col in range(3)
    ):
        continue

    event_mask = (
        JOINT_VAL_10[idx] > 0
    )

    if not event_mask.any():
        continue

    sampled_joint_truth = JOINT_VAL_10[idx]

    sampled_true_probs = JOINT_P_10[
        idx,
        sampled_joint_truth
    ]

    bootstrap_10["Joint_NLL"].append(
        float(
            -np.log(
                np.clip(
                    sampled_true_probs,
                    1e-30,
                    1
                )
            ).mean()
        )
    )

    bootstrap_10["Family_Macro_F1"].append(
        float(
            f1_score(
                FAMILY_VAL_10[idx][event_mask],
                pred_family_all_10[idx][event_mask],
                labels=[1, 2, 3, 4, 5],
                average="macro",
                zero_division=0
            )
        )
    )

    bootstrap_10["ETA_MAE"].append(
        float(
            np.mean(
                np.abs(
                    EXPECTED_ETA_10[idx][event_mask]
                    - ONSET_VAL_10[idx][event_mask]
                )
            )
        )
    )

    for col, horizon in enumerate(HORIZONS_10):
        bootstrap_10[f"H{horizon}_AP"].append(
            float(
                average_precision_score(
                    Y_VAL_10[idx, col],
                    HORIZON_P_10[idx, col]
                )
            )
        )

BOOTSTRAP_CI_10 = {}

for metric, values in bootstrap_10.items():

    if len(values) < 250:
        raise RuntimeError(
            f"Insufficient bootstrap replicates: {metric}"
        )

    lower, upper = np.percentile(
        values,
        [2.5, 97.5]
    )

    BOOTSTRAP_CI_10[metric] = {
        "lower_95": float(lower),
        "upper_95": float(upper),
        "successful_replicates": int(len(values))
    }

# ==========================================================================================
# 10P — COMPARE WITH EXISTING VALIDATED BASELINES
#
# These are previously recorded OFFICIAL VAL metrics.
# They are descriptive reference scores.
#
# No other models are loaded or modified.
# ==========================================================================================

BASELINE_REFERENCE_10 = {
    "XGBoost_V2": {
        "H5_AP": 0.676304,
        "H10_AP": 0.875130,
        "H15_AP": None
    },
    "Stacked_LSTM_V2": {
        "H5_AP": 0.761005,
        "H10_AP": 0.906018,
        "H15_AP": 0.935854
    }
}

comparison_rows_10 = []

for col, horizon in enumerate(HORIZONS_10):

    current_ap = float(
        OVERALL_DF_10.iloc[col]["AP"]
    )

    for baseline_name, baseline_values in (
        BASELINE_REFERENCE_10.items()
    ):

        baseline_ap = baseline_values[
            f"H{horizon}_AP"
        ]

        if baseline_ap is None:
            continue

        comparison_rows_10.append({
            "Horizon": f"H{horizon}",
            "CompetingRisk_LSTM_AP": current_ap,
            "Baseline": baseline_name,
            "Baseline_AP": baseline_ap,
            "Difference": float(
                current_ap - baseline_ap
            )
        })

COMPARISON_DF_10 = pd.DataFrame(
    comparison_rows_10
)

# ==========================================================================================
# 10Q — SAVE WINDOW-LEVEL PREDICTIONS
# ==========================================================================================

PRED_DF_10 = pd.DataFrame({
    "Episode_ID": EPISODE_VAL_10,
    "Minute": MINUTE_VAL_10,
    "Source": SOURCE_VAL_10,
    "True_Joint_Class": JOINT_VAL_10,
    "True_Onset_Minute": ONSET_VAL_10,
    "True_Family_ID": FAMILY_VAL_10,
    "True_H5": Y_VAL_10[:, 0],
    "True_H10": Y_VAL_10[:, 1],
    "True_H15": Y_VAL_10[:, 2],
    "P_No_Event_Within_15": NO_EVENT_P_10,
    "P_H5": HORIZON_P_10[:, 0],
    "P_H10": HORIZON_P_10[:, 1],
    "P_H15": HORIZON_P_10[:, 2],
    "Predicted_Family_If_Event": pred_family_all_10,
    "Predicted_Onset_Minute_If_Event": PREDICTED_MINUTE_10,
    "Expected_ETA_If_Event": EXPECTED_ETA_10,
    "True_Joint_Class_Probability": true_joint_prob_10
})

# Preserve complete joint distribution for later audits.
# Use NPZ so all 76 probabilities are retained at full precision.
PREDICTION_ARRAYS_10 = {
    "joint_probabilities": JOINT_P_10,
    "episode_id": EPISODE_VAL_10,
    "minute": MINUTE_VAL_10,
    "joint_true": JOINT_VAL_10,
    "horizon_true": Y_VAL_10,
    "onset_true": ONSET_VAL_10,
    "family_true": FAMILY_VAL_10,
    "horizon_probabilities": HORIZON_P_10
}

assert len(PRED_DF_10) == 18839

assert not PRED_DF_10.duplicated(
    ["Episode_ID", "Minute"]
).any()

# ==========================================================================================
# 10R — CREATE OFFICIAL VALIDATION REPORT
# ==========================================================================================

REPORT_10 = {
    "experiment": "LSTM_V2_COMPETING_RISK_OFFICIAL_VALIDATION",
    "model": "LSTM_V2_COMPETING_RISK",
    "selected_configuration": "C01",
    "dataset": "Synthetic_NAF_v2",
    "dataset_version": "2.0.0",
    "model_sha256": model_hash_10,
    "validation_data_sha256": expected_val_hash_10,
    "split": "VALIDATION",
    "windows": int(len(JOINT_VAL_10)),
    "episodes": int(len(unique_episodes_10)),
    "joint_classes": 76,
    "joint_nll": JOINT_NLL_10,
    "joint_top1_accuracy": JOINT_TOP1_10,
    "horizon_metrics": overall_rows_10,
    "family_metrics": {
        "evaluation_condition": "TRUE_ONSET_WITHIN_15_MINUTES",
        "family_accuracy": FAMILY_ACCURACY_10,
        "family_macro_f1": FAMILY_MACRO_F1_10,
        "event_windows": int(TRUE_EVENT_MASK_10.sum())
    },
    "onset_metrics": {
        "evaluation_condition": "TRUE_ONSET_WITHIN_15_MINUTES",
        "expected_eta_mae": ETA_MAE_10,
        "expected_eta_rmse": ETA_RMSE_10,
        "exact_minute_accuracy": EXACT_MINUTE_ACCURACY_10,
        "within_one_minute_accuracy": WITHIN_ONE_MINUTE_10,
        "within_two_minutes_accuracy": WITHIN_TWO_MINUTES_10
    },
    "probability_consistency": {
        "H5_greater_than_H10": violations_5_10,
        "H10_greater_than_H15": violations_10_15
    },
    "fixed_thresholds": FIXED_THRESHOLDS_10,
    "bootstrap_95_ci": BOOTSTRAP_CI_10,
    "bootstrap_resampling_unit": "EPISODE",
    "baseline_reference": BASELINE_REFERENCE_10,
    "no_training": True,
    "no_hyperparameter_tuning": True,
    "no_recalibration": True,
    "official_test_accessed": False,
    "frozen_dataset_modified": False,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "evaluated_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "limitations": [
        "Synthetic data, not demonstrated live-network performance.",
        "Episode windows overlap and are statistically dependent.",
        "Episode bootstrap does not fully model calibration-group dependence.",
        "Family and ETA diagnostics condition on true future event.",
        "End-to-end results depend on predeclared fixed thresholds.",
        "Baseline AP comparison is descriptive, not a paired significance test.",
        "Exact-minute classification is a harder task than horizon forecasting."
    ]
}

# Create directory only after all metrics and checks succeed.
REPORT_DIR_10.mkdir(
    parents=True,
    exist_ok=False
)

def save_csv_10(df, filename):
    df.to_csv(
        REPORT_DIR_10 / filename,
        index=False,
        float_format="%.17g"
    )


save_csv_10(
    OVERALL_DF_10,
    "HORIZON_METRICS.csv"
)

save_csv_10(
    CALIBRATION_DF_10,
    "CALIBRATION_10_BINS.csv"
)

save_csv_10(
    FAMILY_PER_CLASS_DF_10,
    "FAMILY_PER_CLASS_METRICS.csv"
)

save_csv_10(
    FAMILY_CM_DF_10,
    "FAMILY_CONFUSION_MATRIX.csv"
)

save_csv_10(
    ETA_BY_MINUTE_DF_10,
    "ETA_BY_TRUE_MINUTE.csv"
)

save_csv_10(
    THRESHOLD_DF_10,
    "FIXED_THRESHOLD_METRICS.csv"
)

save_csv_10(
    END_TO_END_DF_10,
    "END_TO_END_FAMILY_METRICS.csv"
)

save_csv_10(
    BENIGN_DF_10,
    "BENIGN_FALSE_ALERTS.csv"
)

save_csv_10(
    SOURCE_DF_10,
    "SOURCE_HORIZON_METRICS.csv"
)

save_csv_10(
    SOURCE_FAMILY_DF_10,
    "SOURCE_FAMILY_ETA_METRICS.csv"
)

save_csv_10(
    COMPARISON_DF_10,
    "BASELINE_AP_COMPARISON.csv"
)

save_csv_10(
    PRED_DF_10,
    "VALIDATION_PREDICTIONS.csv"
)

np.savez_compressed(
    REPORT_DIR_10 / "JOINT_76_CLASS_VALIDATION_PREDICTIONS.npz",
    **PREDICTION_ARRAYS_10
)

write_json_10(
    REPORT_DIR_10 / "BOOTSTRAP_CONFIDENCE_INTERVALS.json",
    BOOTSTRAP_CI_10
)

write_json_10(
    REPORT_DIR_10 / "OFFICIAL_VALIDATION_REPORT.json",
    REPORT_10
)

# Integrity manifest of completed report artifacts.
artifact_hashes_10 = {
    path.name: sha256_file_10(path)
    for path in sorted(REPORT_DIR_10.iterdir())
    if path.is_file()
}

write_json_10(
    REPORT_DIR_10 / "ARTIFACT_SHA256SUMS.json",
    artifact_hashes_10
)

# ==========================================================================================
# 10S — FINAL CONSOLE REPORT
# ==========================================================================================

print("\n" + "=" * 110)
print("COMPETING-RISK LSTM — OFFICIAL VALIDATION COMPLETE")
print("=" * 110)

print("\nJOINT DISTRIBUTION")
print(f"Joint 76-class NLL       : {JOINT_NLL_10:.6f}")
print(f"Joint top-1 accuracy     : {JOINT_TOP1_10:.6f}")

print("\nHORIZON FORECASTING")
print(
    OVERALL_DF_10.round(6).to_string(index=False)
)

print("\nCONDITIONAL ATTACK-FAMILY PERFORMANCE")
print(f"Family accuracy         : {FAMILY_ACCURACY_10:.6f}")
print(f"Family macro-F1         : {FAMILY_MACRO_F1_10:.6f}")
print(f"Event-positive windows  : {TRUE_EVENT_MASK_10.sum()}")

print("\nPER-FAMILY METRICS")
print(
    FAMILY_PER_CLASS_DF_10.round(6).to_string(index=False)
)

print("\nFAMILY CONFUSION MATRIX")
print(FAMILY_CM_DF_10.to_string())

print("\nCONDITIONAL ONSET-TIME PERFORMANCE")
print(f"Expected ETA MAE        : {ETA_MAE_10:.6f} minutes")
print(f"Expected ETA RMSE       : {ETA_RMSE_10:.6f} minutes")
print(f"Exact-minute accuracy   : {EXACT_MINUTE_ACCURACY_10:.6f}")
print(f"Within 1 minute         : {WITHIN_ONE_MINUTE_10:.6f}")
print(f"Within 2 minutes        : {WITHIN_TWO_MINUTES_10:.6f}")

print("\nFIXED-THRESHOLD ATTACK DETECTION")
print(
    THRESHOLD_DF_10.round(6).to_string(index=False)
)

print("\nEND-TO-END FAMILY FORECASTING")
print(
    END_TO_END_DF_10.round(6).to_string(index=False)
)

print("\nBENIGN FALSE ALERTS")
print(
    BENIGN_DF_10.round(6).to_string(index=False)
)

print("\nSOURCE-SPECIFIC HORIZON FORECASTING")
print(
    SOURCE_DF_10.round(6).to_string(index=False)
)

print("\nSOURCE-SPECIFIC FAMILY AND ETA")
print(
    SOURCE_FAMILY_DF_10.round(6).to_string(index=False)
)

print("\nEPISODE-BOOTSTRAP 95% CONFIDENCE INTERVALS")

for metric, ci in BOOTSTRAP_CI_10.items():
    print(
        f"{metric:20s}: "
        f"[{ci['lower_95']:.6f}, {ci['upper_95']:.6f}]"
    )

print("\nPREVIOUS MODEL COMPARISON")
print(
    COMPARISON_DF_10.round(6).to_string(index=False)
)

print("\nPROBABILITY CONSISTENCY")
print("P(H5) > P(H10) violations:", violations_5_10)
print("P(H10) > P(H15) violations:", violations_10_15)

print("\nSAVED OFFICIAL REPORTS")
print(REPORT_DIR_10)

print("\nModel SHA256:", model_hash_10)

print("\nModel weights: UNCHANGED")
print("Additional training: NONE")
print("Recalibration: NONE")
print("Official TEST: NOT ACCESSED")
print("Frozen dataset: UNCHANGED")

print("\n" + "=" * 110)
print("OFFICIAL COMPETING-RISK VALIDATION COMPLETED SUCCESSFULLY")
print("=" * 110)
